# Person 1 — corrected data workflow v2

Supports the REPEAT/RANDOM/FIT mathematical-reasoning experiments with traceable worked-solution inputs. The v1 artifacts are historical and contain confirmed bad targets. This notebook has been changed and its old outputs cleared.

Run into a new immutable `full_run_v2_*` directory. Review CSVs retain completed decisions on identical inputs. All manifests are drafts until the final evidence gate passes. Missing development sets, actual labels, or actual parent evidence cannot be replaced by templates. See `capstone_person1/FIXES_AND_HANDOFF.md`.


In [ ]:
%pip install -q "datasets==5.1.0" "transformers==4.57.6" "tokenizers==0.22.2" "datasketch==2.0.0" "huggingface_hub==0.36.0" "pandas>=2.2" "pyarrow>=17" "tqdm>=4.66"


2 Connect Google Drive and create the project folders

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import tempfile
import uuid

if os.environ.get('CAPSTONE_MOUNT_DRIVE', '0') == '1':
    from google.colab import drive
    drive.mount('/content/drive')
    default_root = Path('/content/drive/MyDrive/capstone_person1')
else:
    default_root = Path.cwd() / 'capstone_person1'
PROJECT_ROOT = Path(os.environ.get('CAPSTONE_PROJECT_ROOT', str(default_root))).expanduser().resolve()

# New runs never overwrite v1. To resume reviews, set this to the SAME v2 run ID
# and RESUME_RUN=True; all frozen configuration and review inputs must still match.
RUN_FOLDER_NAME = os.environ.get('CAPSTONE_RUN_ID') or (
    'full_run_v2_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
)
RESUME_RUN = False
if not RUN_FOLDER_NAME.startswith('full_run_v2_') or Path(RUN_FOLDER_NAME).name != RUN_FOLDER_NAME:
    raise ValueError('Use a single full_run_v2_* folder name')
RUN_ROOT = PROJECT_ROOT / 'runs' / RUN_FOLDER_NAME
if RUN_ROOT.exists() and not RESUME_RUN:
    raise FileExistsError(f'Immutable run already exists: {RUN_ROOT}. Choose a new ID or explicitly resume.')
if RESUME_RUN and not (RUN_ROOT / 'run_configuration.json').exists():
    raise ValueError('Resume requires a previously frozen v2 configuration')
RUN_ROOT.mkdir(parents=True, exist_ok=RESUME_RUN)
INPUT_DIR = PROJECT_ROOT / 'inputs_from_team'
OUTPUT_DIR = RUN_ROOT / 'outputs'
REPORT_DIR = OUTPUT_DIR / 'reports'
MANIFEST_DIR = OUTPUT_DIR / 'manifests'
PROCESSED_DIR = OUTPUT_DIR / 'processed'
LOCAL_WORK_DIR = Path(tempfile.gettempdir()) / ('capstone_' + RUN_FOLDER_NAME)
HF_CACHE_DIR = PROJECT_ROOT / 'cache' / 'huggingface'
for folder in [INPUT_DIR, REPORT_DIR, MANIFEST_DIR, PROCESSED_DIR, LOCAL_WORK_DIR, HF_CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print('Run folder:', RUN_ROOT)
print('All exports remain DRAFT until the final saved-evidence approval cell passes.')


3 Freeze the project configuration

In [ ]:
# -----------------------------
# EDIT ONLY THIS CONFIGURATION
# -----------------------------
RUN_MODE = 'full'             # 'pilot' first; 'full' for final artifacts
PILOT_ROWS_PER_DATASET = 25_000

ORCA_DATASET_ID = 'microsoft/orca-math-word-problems-200k'
NUMINA_DATASET_ID = 'AI-MO/NuminaMath-1.5'
TOKENIZER_ID = 'Qwen/Qwen2.5-0.5B'
# Fixed SHAs from the inspected v1 run. Updating a SHA requires a new run.
ORCA_REVISION = '29255d1770cc4eac66e5e7fa378cba542c026350'
NUMINA_REVISION = '1b05109f9e5c1ad06c0663519502416c30b300f8'
TOKENIZER_REVISION = '060db6499f32faf8b98477b0a26969ef7d8b9987'
GSM8K_REVISION = '740312add88f781978c0658806c59bc2815b9866'
MATH500_REVISION = '6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be'
REVIEW_POLICY = {
    'version': 'person1-review-v2',
    'overlap_scope': 'conservative_lexical_variants',
    'near_duplicate_threshold': 0.90,
    'decontamination_threshold': 0.75,
    'diagram_policy': 'quarantine_external_images_and_inline_asy_tikz',
    'solution_policy': 'worked_solutions_only; quarantine_suspected_answer_only',
    'multi_part_policy': 'quarantine_suspected_multi_part_until_reviewed_full_tuple',
    'correctness_policy': 'unknown_until_independent_mathematical_review',
}
DEV_MANIFEST_PATH = INPUT_DIR / 'frozen_development.csv'
DEV_METADATA_PATH = INPUT_DIR / 'frozen_development_metadata.json'
SHARED_EQUIVALENCE_PATH = INPUT_DIR / 'equivalence_checker.py'
PARENT_METADATA_PATH = INPUT_DIR / 'parent_checkpoint_R1000.json'
PARENT_ACTUAL_MANIFEST_PATH = INPUT_DIR / 'parent_actual_training_manifest_R1000.jsonl'
PERSON2_TOKEN_METADATA_PATH = INPUT_DIR / 'person2_token_verification_metadata.json'
INTEGRATION_BLOCKERS = []
DEVELOPMENT_PROTECTED = False
TOKEN_LABELS_VERIFIED = False
PARENT_CONFIRMED = False

SPLIT_SEED = 1729
SELECTION_SEED = 42
MAX_TOTAL_TOKENS = 2048
MANIFEST_SIZES = [500, 1_000, 2_000, 4_000, 8_000, 16_000, 40_000]
TARGET_WORKING_POOL = 40_000
TARGET_PER_SOURCE = 20_000
CANDIDATE_UNIVERSE_PER_SOURCE = 30_000
FRESH_POOL_TARGET = 10_000

# LSH only generates pairs for review. Exact shingle Jaccard decides removal.
LSH_CANDIDATE_THRESHOLD = 0.65
NEAR_DUPLICATE_THRESHOLD = 0.90
DECONTAMINATION_THRESHOLD = 0.75
MINHASH_PERMUTATIONS = 128
TOKENIZATION_BATCH_SIZE = 256

# Retain the synthetic flag by default. Change only through a recorded team decision.
EXCLUDE_SYNTHETIC_NUMINA = False

EVAL_MANIFEST_PATH = INPUT_DIR / 'reserved_evaluation.csv'
PARENT_SEEN_GROUPS_PATH = INPUT_DIR / 'parent_seen_groups_R1000.csv'

assert RUN_MODE in {'pilot', 'full'}
REQUIRE_EVAL_MANIFEST = RUN_MODE == 'full'
print('Run mode:', RUN_MODE)
print('Final run requires evaluation manifest:', REQUIRE_EVAL_MANIFEST)
# BEGIN STRUCTURED DATA CONFIGURATION
import json
"""Validate the data configuration and map it to notebook stage settings."""
import re


def notebook_settings(data):
    if data['tokenizer_id'] != 'Qwen/Qwen2.5-0.5B' or data['tokenizer_revision'] != '060db6499f32faf8b98477b0a26969ef7d8b9987':
        raise ValueError('This release requires the agreed pinned 0.5B tokenizer')
    for source in data['sources'].values():
        if source['split'] != 'train' or not re.fullmatch(r'[0-9a-f]{40}', source['revision']):
            raise ValueError('Source split/revision must be an immutable train snapshot')
    for key in ['train_bank_per_source', 'selection_bank_per_source', 'loss_dev_per_source', 'max_total_tokens']:
        if type(data[key]) is not int or data[key] < 1: raise ValueError('Invalid positive count: ' + key)
    return {
        'ORCA_DATASET_ID': data['sources']['orca']['dataset_id'], 'NUMINA_DATASET_ID': data['sources']['numina']['dataset_id'],
        'ORCA_REVISION': data['sources']['orca']['revision'], 'NUMINA_REVISION': data['sources']['numina']['revision'],
        'TOKENIZER_ID': data['tokenizer_id'], 'TOKENIZER_REVISION': data['tokenizer_revision'],
        'GSM8K_REVISION': data['reserved_tests']['gsm8k']['revision'], 'MATH500_REVISION': data['reserved_tests']['math500']['revision'],
        'SPLIT_SEED': data['split_seed'], 'SELECTION_SEED': data['selection_seed'], 'MAX_TOTAL_TOKENS': data['max_total_tokens'],
        'TARGET_PER_SOURCE': data['train_bank_per_source'], 'TARGET_WORKING_POOL': data['train_bank_per_source'] * 2,
        'FRESH_POOL_TARGET': data['selection_bank_per_source'] * 2, 'LOSS_DEV_TARGET': data['loss_dev_per_source'] * 2,
        'MANIFEST_SIZES': data['required_training_prefixes'] + data['optional_training_prefixes'],
        'CANDIDATE_UNIVERSE_PER_SOURCE': data['candidate_universe_per_source'],
        'LSH_CANDIDATE_THRESHOLD': data['lsh_candidate_threshold'], 'NEAR_DUPLICATE_THRESHOLD': data['near_duplicate_threshold'],
        'DECONTAMINATION_THRESHOLD': data['decontamination_threshold'], 'MINHASH_PERMUTATIONS': data['minhash_permutations'],
        'EXCLUDE_SYNTHETIC_NUMINA': data['exclude_synthetic_numina'],
    }

DATA_CONFIG = {'configuration_version': 'person1-config-v3', 'notebook': 'Person1_Full_Run_v2.ipynb', 'model_id': 'Qwen/Qwen2.5-0.5B', 'tokenizer_id': 'Qwen/Qwen2.5-0.5B', 'tokenizer_revision': '060db6499f32faf8b98477b0a26969ef7d8b9987', 'sources': {'orca': {'dataset_id': 'microsoft/orca-math-word-problems-200k', 'revision': '29255d1770cc4eac66e5e7fa378cba542c026350', 'split': 'train'}, 'numina': {'dataset_id': 'AI-MO/NuminaMath-1.5', 'revision': '1b05109f9e5c1ad06c0663519502416c30b300f8', 'split': 'train'}}, 'reserved_tests': {'gsm8k': {'dataset_id': 'openai/gsm8k', 'revision': '740312add88f781978c0658806c59bc2815b9866', 'configuration': 'main', 'split': 'test'}, 'math500': {'dataset_id': 'HuggingFaceH4/MATH-500', 'revision': '6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be', 'split': 'test'}}, 'split_seed': 1729, 'selection_seed': 42, 'max_total_tokens': 2048, 'train_bank_per_source': 20000, 'selection_bank_per_source': 5000, 'loss_dev_per_source': 500, 'required_training_prefixes': [500, 1000, 2000, 4000], 'optional_training_prefixes': [8000, 16000, 40000], 'candidate_universe_per_source': 30000, 'intended_overlap_policy': 'instance_or_near_instance', 'legacy_matching_engine_status': 'diagnostic_only_until_instance_policy_alignment', 'lsh_candidate_threshold': 0.65, 'near_duplicate_threshold': 0.9, 'decontamination_threshold': 0.75, 'minhash_permutations': 128, 'exclude_synthetic_numina': False, 'project_root_override_environment': 'CAPSTONE_PROJECT_ROOT', 'mount_drive_environment': 'CAPSTONE_MOUNT_DRIVE', 'default_mount_drive': False, 'person3_development_coverage': 'pending', 'completion_policy': 'preserve_correct_terminal_block; quarantine_contradictions', 'prompt_policy': 'exact_plain_text_no_chat_template', 'expected_eos_token': '<|endoftext|>', 'expected_eos_token_id': 151643, 'expected_pad_token': '<|endoftext|>', 'expected_pad_token_id': 151643, 'supervision_policy': 'completion_and_one_appended_eos; prompt_and_padding_labels=-100'}
DATA_CONFIG_PATH = PROJECT_ROOT / 'configs/person1_data_config.json'
if DATA_CONFIG_PATH.exists():
    DATA_CONFIG = json.loads(DATA_CONFIG_PATH.read_text(encoding='utf-8'))
globals().update(notebook_settings(DATA_CONFIG))
REVIEW_POLICY['near_duplicate_threshold'] = NEAR_DUPLICATE_THRESHOLD
REVIEW_POLICY['decontamination_threshold'] = DECONTAMINATION_THRESHOLD
REVIEW_POLICY['diagram_policy'] = 'quarantine_external_images_and_inline_asy_tikz_svg'
# The old threshold-only engine remains diagnostic until the later matching stage is aligned.
INTEGRATION_BLOCKERS.append('Instance-level duplicate/decontamination policy alignment pending')
# END STRUCTURED DATA CONFIGURATION


In [ ]:
# ============================================================
# CREATE THE RESERVED EVALUATION MANIFEST
# Run this once and then keep the generated file unchanged.
# ============================================================

from datasets import load_dataset
from huggingface_hub import HfApi
from datetime import datetime, timezone
import pandas as pd
import hashlib
import json

EVAL_MANIFEST_PATH = INPUT_DIR / "reserved_evaluation.csv"
EVAL_METADATA_PATH = INPUT_DIR / "reserved_evaluation_metadata.json"

required_columns = {"set_id", "problem_id", "problem"}

# Avoid accidentally replacing the frozen evaluation file.
if EVAL_MANIFEST_PATH.exists():

    existing_eval = pd.read_csv(EVAL_MANIFEST_PATH)

    missing_columns = required_columns - set(existing_eval.columns)
    if missing_columns:
        raise ValueError(
            f"Existing evaluation file is missing columns: {missing_columns}"
        )

    print("Evaluation manifest already exists.")
    print("It was not overwritten.")
    print("Location:", EVAL_MANIFEST_PATH)
    print("Total problems:", len(existing_eval))
    display(existing_eval["set_id"].value_counts())

else:

    api = HfApi()

    # Record the exact Hugging Face dataset revisions.
    gsm8k_revision = GSM8K_REVISION
    math500_revision = MATH500_REVISION

    print("Downloading GSM8K test split...")
    gsm8k_test = load_dataset(
        "openai/gsm8k",
        "main",
        split="test",
        revision=gsm8k_revision
    )

    print("Downloading MATH-500 test split...")
    math500_test = load_dataset(
        "HuggingFaceH4/MATH-500",
        split="test",
        revision=math500_revision
    )

    # The manifest contains only problem text.
    # Evaluation answers are intentionally not saved here.
    gsm8k_rows = pd.DataFrame({
        "set_id": ["gsm8k_test"] * len(gsm8k_test),
        "problem_id": [
            f"gsm8k_test_{i:04d}"
            for i in range(len(gsm8k_test))
        ],
        "problem": [
            str(question).strip()
            for question in gsm8k_test["question"]
        ]
    })

    math500_rows = pd.DataFrame({
        "set_id": ["math500_test"] * len(math500_test),
        "problem_id": [
            f"math500_test_{i:04d}"
            for i in range(len(math500_test))
        ],
        "problem": [
            str(problem).strip()
            for problem in math500_test["problem"]
        ]
    })

    reserved_evaluation = pd.concat(
        [gsm8k_rows, math500_rows],
        ignore_index=True
    )

    # Validation checks
    assert len(gsm8k_rows) == 1319, (
        f"Expected 1,319 GSM8K test problems, "
        f"found {len(gsm8k_rows)}"
    )

    assert len(math500_rows) == 500, (
        f"Expected 500 MATH-500 problems, "
        f"found {len(math500_rows)}"
    )

    assert len(reserved_evaluation) == 1819
    assert reserved_evaluation["problem_id"].is_unique
    assert reserved_evaluation["problem"].str.len().gt(0).all()

    # Save the frozen manifest.
    reserved_evaluation.to_csv(
        EVAL_MANIFEST_PATH,
        index=False
    )

    file_hash = hashlib.sha256(
        EVAL_MANIFEST_PATH.read_bytes()
    ).hexdigest()

    metadata = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "total_rows": len(reserved_evaluation),
        "set_counts": (
            reserved_evaluation["set_id"]
            .value_counts()
            .to_dict()
        ),
        "datasets": {
            "gsm8k": {
                "dataset_id": "openai/gsm8k",
                "configuration": "main",
                "split": "test",
                "revision": gsm8k_revision
            },
            "math500": {
                "dataset_id": "HuggingFaceH4/MATH-500",
                "split": "test",
                "revision": math500_revision
            }
        },
        "manifest_sha256": file_hash
    }

    with open(EVAL_METADATA_PATH, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    print("Reserved evaluation manifest created successfully.")
    print("Manifest:", EVAL_MANIFEST_PATH)
    print("Metadata:", EVAL_METADATA_PATH)
    print("Total problems:", len(reserved_evaluation))
    print("SHA-256:", file_hash)

    display(reserved_evaluation["set_id"].value_counts())
    display(reserved_evaluation.head())

4 Import libraries and record the environment

In [ ]:
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import random
import re
import shutil
import sys
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from datasets import Dataset, concatenate_datasets, load_dataset
from datasketch import MinHash, MinHashLSH
from huggingface_hub import HfApi
from tqdm.auto import tqdm
from transformers import AutoTokenizer

CONTROL_SOURCE = '"""Versioned, offline-testable controls embedded in Person1_Full_Run_v2.ipynb.\n\nThese checks establish parsing/structural properties, not mathematical correctness.\n"""\nimport ast\nimport hashlib\nimport json\nimport math\nimport re\nimport unicodedata\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nfrom fractions import Fraction\nfrom pathlib import Path\n\nCONTROL_VERSION = \'person1-v3.0\'\nPROMPT_SPEC_VERSION = \'math-worked-solution-v2\'\nPROMPT_TEMPLATE = (\n    \'Question: {problem}\\n\'\n    \'Solve the problem step by step. End with: Final answer: <answer>.\\n\'\n    \'Solution:\\n\'\n)\nNUMBER_PATTERN = r\'[+-]?(?:(?:\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.\\d+)?|\\.\\d+)(?:\\s*/\\s*[+-]?(?:\\d+(?:\\.\\d+)?|\\.\\d+))?%?\'\nNUMBER_RE = re.compile(NUMBER_PATTERN)\nSUSPECT_ANSWER_RE = re.compile(r\'\\(,|,\\)|,,|,=|^,|,$|^=|\\(\\)\')\nCJK_RE = re.compile(r\'[\\u4e00-\\u9fff]\')\n\n\ndef clean_text(value):\n    if value is None or type(value).__name__ in {\'NAType\', \'NaTType\'}:\n        return \'\'\n    if not isinstance(value, str):\n        try:\n            if math.isnan(value): return \'\'\n        except (TypeError, ValueError): pass\n    text = unicodedata.normalize(\'NFKC\', str(value)).replace(\'\\r\\n\', \'\\n\').replace(\'\\r\', \'\\n\')\n    return re.sub(r\'\\n{3,}\', \'\\n\\n\', re.sub(r\'[ \\t]+\', \' \', text)).strip()\n\n\ndef raw_text(value):\n    """Preserve original string bytes; represent missing scalar text as empty."""\n    return value if isinstance(value, str) else clean_text(value)\n\n\ndef compact_answer(value):\n    # Never silently truncate an answer (v1 truncated at 500 characters).\n    text = clean_text(value)\n    return text[:-1].rstrip() if text.endswith(\'.\') else text\n\n\ndef normalize_problem_for_matching(value):\n    return re.sub(r\'\\s+\', \' \', clean_text(value).casefold()).strip()\n\n\ndef sha256_text(value):\n    return hashlib.sha256(value.encode(\'utf-8\')).hexdigest()\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef squash_answer(value):\n    return re.sub(r\'\\s+|\\\\left|\\\\right|\\$|\\\\!|\\\\,|\\\\;|\\\\displaystyle\', \'\', compact_answer(value).casefold())\n\n\ndef scalar_fraction(value):\n    """Safe exact arithmetic only; no eval, symbolic parsing or approximate equality."""\n    text = squash_answer(value)\n    text = re.sub(r\'^([a-z])=\', \'\', text)\n    text = re.sub(r\'\\\\(?:dfrac|tfrac|frac)\\{([+-]?\\d+)\\}\\{([+-]?\\d+)\\}\', r\'(\\1/\\2)\', text)\n    # Only thousands separators are numeric commas; (1,2) is not scalar 12.\n    comma_checked = re.sub(r\'(?<!\\d)\\d{1,3}(?:,\\d{3})+(?!\\d)\', lambda match: match.group().replace(\',\', \'\'), text)\n    if \',\' in comma_checked: return None\n    text = comma_checked\n    percent = text.endswith(\'%\')\n    if percent:\n        text = text[:-1]\n    if not text or len(text) > 100 or not re.fullmatch(r\'[\\d.+*/() -]+\', text):\n        return None\n    def visit(node):\n        if isinstance(node, ast.Constant) and type(node.value) in (int, float):\n            return Fraction(ast.get_source_segment(text, node))\n        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):\n            result = visit(node.operand)\n            return result if isinstance(node.op, ast.UAdd) else -result\n        if isinstance(node, ast.BinOp):\n            left, right = visit(node.left), visit(node.right)\n            if isinstance(node.op, ast.Add): return left + right\n            if isinstance(node.op, ast.Sub): return left - right\n            if isinstance(node.op, ast.Mult): return left * right\n            if isinstance(node.op, ast.Div): return left / right\n        raise ValueError(\'unsupported scalar\')\n    try:\n        value = visit(ast.parse(text, mode=\'eval\').body)\n        return value / 100 if percent else value\n    except (ValueError, SyntaxError, ZeroDivisionError, TypeError):\n        return None\n\n\ndef normalize_number(value):\n    fraction = scalar_fraction(value)\n    if fraction is None:\n        return value\n    return str(fraction.numerator) if fraction.denominator == 1 else f\'{fraction.numerator}/{fraction.denominator}\'\n\n\ndef answer_key(value):\n    fraction = scalar_fraction(value)\n    return \'rational:\' + str(fraction) if fraction is not None else \'text:\' + squash_answer(value)\n\n\ndef answers_equivalent(left, right, shared_checker=None):\n    """True/False for exact rational scalars; None means equivalence is unproven."""\n    if squash_answer(left) == squash_answer(right):\n        return True\n    a, b = scalar_fraction(left), scalar_fraction(right)\n    if a is not None and b is not None:\n        return a == b\n    if shared_checker is not None:\n        result = shared_checker(left, right)\n        if result not in (True, False, None):\n            raise ValueError(\'Shared checker must return True, False or None\')\n        return result\n    return None\n\n\ndef suspected_multi_part(problem):\n    text = clean_text(problem).casefold()\n    # Conservative scope: multi-part examples await a reviewed full answer tuple.\n    return bool(re.search(\n        r\'\\b(?:both|respectively)\\b|\\b(?:radius|diameter|length|width|area|perimeter|volume)\'\n        r\'\\s+(?:and|,)\\s+(?:the\\s+)?(?:radius|diameter|length|width|area|perimeter|volume)\\b\'\n        r\'|\\bhow much did each\\b|\\b(?:find|calculate|determine|what are)\\b[^?\\n]*\\band\\b\'\n        r\'|(?:\\(a\\)|\\(b\\)|\\bi\\)|\\bii\\))\', text))\n\n\ndef boxed_spans(solution):\n    spans = []\n    for marker in re.finditer(r\'\\\\(?:boxed|fbox)\\s*\\{\', solution):\n        brace, depth = marker.end() - 1, 0\n        for pos in range(brace, len(solution)):\n            if solution[pos] == \'{\': depth += 1\n            elif solution[pos] == \'}\':\n                depth -= 1\n                if depth == 0:\n                    spans.append((marker.start(), pos + 1, compact_answer(solution[brace + 1:pos])))\n                    break\n    return spans\n\n\ndef last_boxed(solution):\n    spans = boxed_spans(solution or \'\')\n    return spans[-1][2] if spans else \'\'\n\n\nFINAL_MARKER_RE = re.compile(r\'(?im)^[ \\t]*final answer[ \\t]*[:=]\')\nTERMINAL_MARKER_RE = re.compile(r\'(?im)^[ \\t]*(final answer[ \\t]*[:=]|####)[ \\t]*([^\\n]+?)[ \\t]*(?:\\n[ \\t]*)*\\Z\')\nMISSING_ANSWER_VALUES = {\'\', \'proof\', \'notfound\', \'nan\', \'none\', \'null\', \'<na>\', \'nat\', \'n/a\'}\n\n\ndef answer_type(value):\n    """Bounded shape validation; supported symbolic syntax is not a correctness proof."""\n    text = compact_answer(value)\n    if text.casefold() in MISSING_ANSWER_VALUES: return \'missing\'\n    if len(text) > 2000: return \'unsupported_oversized\'\n    if re.search(r\'\\b(?:or|and|respectively)\\b|\\\\pm|±\', text, re.I): return \'unsupported_multi_answer\'\n    without_numeric_commas = re.sub(r\'\\d{1,3}(?:,\\d{3})+\', \'\', text)\n    if \',\' in without_numeric_commas or \';\' in text: return \'unsupported_multi_answer\'\n    if scalar_fraction(text) is not None: return \'numeric_scalar\'\n    # An equality chain must be reduced/reviewed explicitly, never accepted as its first number.\n    if \'=\' in text and not re.fullmatch(r\'[a-zA-Z]\\s*=\\s*[^=]+\', text):\n        return \'unsupported_equality_chain\'\n    if re.fullmatch(r\'\\$?\' + NUMBER_PATTERN + r\'\\s+[A-Za-z][A-Za-z²³^ /-]*\', text) and not re.search(\n            r\'\\b(?:times|plus|minus|divided|pi|sqrt)\\b\', text, re.I):\n        return \'quantity_scalar\'\n    core = text.strip(\'$\')\n    for left, right in [(r\'\\(\', r\'\\)\'), (r\'\\[\', r\'\\]\')]:\n        if core.startswith(left) and core.endswith(right): core = core[len(left):-len(right)]\n    stack = []\n    for char in core:\n        if char in \'({[\': stack.append(char)\n        elif char in \')}]\':\n            if not stack or stack.pop() != {\')\':\'(\', \'}\':\'{\', \']\':\'[\'}[char]: return \'unsupported_unbalanced\'\n    if stack: return \'unsupported_unbalanced\'\n    commands = re.findall(r\'\\\\([A-Za-z]+)\', core)\n    allowed = {\'frac\',\'dfrac\',\'tfrac\',\'sqrt\',\'pi\',\'alpha\',\'beta\',\'gamma\',\'theta\',\'lambda\',\'sigma\',\'omega\',\n               \'infty\',\'sin\',\'cos\',\'tan\',\'log\',\'ln\',\'exp\',\'left\',\'right\',\'cdot\',\'times\',\'boxed\',\'fbox\'}\n    if any(command not in allowed for command in commands): return \'unsupported_symbolic_command\'\n    residue = re.sub(r\'\\\\[A-Za-z]+\', \'\', core)\n    if not re.fullmatch(r\'[A-Za-z0-9\\s.+*/^_(){}\\[\\]=\\-√π]+\', residue): return \'unsupported_structure\'\n    identifiers = re.findall(r\'[A-Za-z]+\', residue)\n    if any(len(name) > 1 and name not in {\'sqrt\',\'pi\',\'sin\',\'cos\',\'tan\',\'log\',\'ln\',\'exp\',\'abs\'} for name in identifiers):\n        return \'unsupported_prose\'\n    if not commands and not identifiers and not re.search(r\'[√π]\', core): return \'unsupported_numeric_expression\'\n    return \'symbolic_scalar\'\n\n\n@dataclass(frozen=True)\nclass AnswerExtraction:\n    candidate: str\n    mode: str\n    evidence_text: str = \'\'\n    evidence_start: int = -1\n    evidence_end: int = -1\n    confidence: str = \'none\'\n    ambiguity_reason: str = \'\'\n    answer_type: str = \'missing\'\n    evidence_basis: str = \'canonical_solution\'\n\n    def to_dict(self): return asdict(self)\n\n\ndef extract_orca_final_answer(problem, solution):\n    """Only the terminal conclusion can supply a target. Never scan earlier givens."""\n    solution = clean_text(solution)\n    if not solution:\n        return \'\', \'missing\'\n    if suspected_multi_part(problem):\n        return \'\', \'ambiguous_multi_part\'\n    # Explicit markers must form the trailing block; earlier markers are ignored.\n    markers = list(FINAL_MARKER_RE.finditer(solution))\n    marker = TERMINAL_MARKER_RE.search(solution)\n    if len(markers) > 1: return \'\', \'ambiguous_multiple_final_markers\'\n    if markers and marker is None: return \'\', \'ambiguous_nonterminal_final_marker\'\n    if marker:\n        value = compact_answer(marker.group(2))\n        kind = answer_type(value)\n        if kind == \'missing\':\n            return \'\', \'missing\'\n        if kind.startswith(\'unsupported\'): return \'\', \'ambiguous_\' + kind\n        return value, \'explicit_final_marker\' if marker.group(1) != \'####\' else \'gsm_marker\'\n    boxes = boxed_spans(solution)\n    if boxes:\n        # A box inside the derivation is not a final answer; multiple boxes need review.\n        if len(boxes) > 1:\n            return \'\', \'ambiguous_multiple_boxes\'\n        _, end, value = boxes[-1]\n        trailing = solution[end:]\n        if re.fullmatch(r\'[\\s.$\\\\\\[\\]()!]*\', trailing):\n            return (value, \'boxed\') if not answer_type(value).startswith(\'unsupported\') else (\'\', \'ambiguous_boxed_structure\')\n    paragraph = solution.split(\'\\n\\n\')[-1]\n    sentence = re.split(r\'(?<=[.!?])\\s+(?=[A-Z])\', paragraph)[-1].strip()\n    # Anchored answer clause, e.g. \'6 is subtracted ... result would be 6\'.\n    clause = re.search(\n        r\'(?i)\\b(?:answer|result|number|value|total|cost|price|distance|area|radius|amount)\'\n        r\'(?:\\s+\\w+){0,5}?\\s+(?:is|equals|would be|will be)\\s*[:=]?\\s*\\$?(\'\n        + NUMBER_PATTERN + r\')(?:\\s+[A-Za-z][A-Za-z²³^ /-]*)?[.!]?$\', sentence)\n    if clause:\n        # Numeric clauses with operator words require explicit review, not unit stripping.\n        if re.search(r\'\\b(?:times|plus|minus|divided|pi|sqrt)\\b\', sentence[clause.start(1):], re.I):\n            return \'\', \'ambiguous_unmarked_math_expression\'\n        return normalize_number(clause.group(1)), \'terminal_answer_clause\'\n    if re.search(r\'\\\\(?:sqrt|frac|dfrac|tfrac|pi)\\b|[√π]|[\\w)]\\s*\\^|\\b[a-zA-Z]\\s*[+\\-]\\s*\\d\', sentence):\n        return \'\', \'ambiguous_unmarked_math_expression\'\n    numbers = NUMBER_RE.findall(sentence)\n    # Repeated numbers and arithmetic expressions remain ambiguous, never select \'novel\' ones.\n    if len(numbers) == 1:\n        return normalize_number(numbers[0]), \'terminal_single_number_review\'\n    return \'\', \'ambiguous_terminal\' if numbers else \'missing\'\n\n\ndef extract_answer_metadata(problem, solution):\n    text = clean_text(solution)\n    answer, mode = extract_orca_final_answer(problem, text)\n    marker = TERMINAL_MARKER_RE.search(text)\n    boxes = boxed_spans(text)\n    if marker:\n        start, end = marker.span()\n    elif mode == \'boxed\' and boxes:\n        start, end, _ = boxes[-1]\n    else:\n        start = text.rfind(\'\\n\\n\') + 2 if \'\\n\\n\' in text else 0\n        end = len(text)\n    confidence = \'marked_terminal\' if mode in {\'explicit_final_marker\',\'gsm_marker\',\'boxed\'} else (\n        \'terminal_clause\' if mode == \'terminal_answer_clause\' else \'heuristic\' if answer else \'none\')\n    return AnswerExtraction(answer, mode, text[start:end], start, end, confidence,\n                            mode.removeprefix(\'ambiguous_\') if mode.startswith(\'ambiguous\') else \'\', answer_type(answer))\n\n\ndef extraction_consistency_status(extraction, final_answer):\n    if not extraction.candidate or extraction.confidence == \'heuristic\':\n        return \'unresolved_requires_review\'\n    equivalent = answers_equivalent(extraction.candidate, final_answer)\n    return \'consistent_terminal_evidence\' if equivalent is True else \'contradictory\' if equivalent is False else \'unresolved_requires_review\'\n\n\ndef diagram_flags(problem, solution):\n    text = f\'{problem}\\n{solution}\'.casefold()\n    flags = []\n    if re.search(r\'\\[/?asy\\]|\\\\begin\\{(?:asy|tikzpicture)\\}|<svg\\b\', text):\n        flags.append(\'inline_diagram_code\')\n    if re.search(r\'!\\[[^\\]]*\\]\\s*(?:\\(|\\[)|<img\\b|\\\\includegraphics\\b\'\n                 r\'|\\b(?:figure|diagram)\\s+(?:below|above)|shown in the (?:figure|diagram)\', text):\n        flags.append(\'missing_or_external_diagram\')\n    return flags\n\n\ndef has_image_reference(problem, solution):\n    return bool(diagram_flags(problem, solution))\n\n\ndef looks_answer_only(solution, final_answer):\n    text = clean_text(solution)\n    if answers_equivalent(text, final_answer) is True:\n        return True\n    prose = re.sub(r\'\\\\[a-zA-Z]+|[^a-zA-Z ]\', \' \', text)\n    words = prose.split()\n    return len(words) < 8 and len(text.splitlines()) <= 2 and text.count(\'=\') <= 1\n\n\ndef target_quality_flags(problem, solution, final_answer, answer_status):\n    flags = []\n    if answer_status.startswith(\'ambiguous\'):\n        flags.append(\'ambiguous_final_answer\')\n    kind = answer_type(final_answer)\n    if kind == \'missing\': flags.append(\'missing_or_sentinel_final_answer\')\n    elif kind.startswith(\'unsupported\'): flags.append(\'unsupported_final_answer_structure\')\n    if answer_status == \'terminal_single_number_review\':\n        flags.append(\'heuristic_terminal_answer_requires_individual_review\')\n    if suspected_multi_part(problem):\n        flags.append(\'suspected_multi_part_requires_review\')\n    if solution and looks_answer_only(solution, final_answer):\n        flags.append(\'suspected_answer_only_solution\')\n    # A recognized terminal scalar disagreement is a separate consistency defect.\n    terminal, status = extract_orca_final_answer(problem, solution)\n    if terminal and status in {\'explicit_final_marker\', \'gsm_marker\', \'boxed\', \'terminal_answer_clause\'}:\n        if answers_equivalent(terminal, final_answer) is False:\n            flags.append(\'terminal_answer_disagrees_with_target\')\n    markers = list(FINAL_MARKER_RE.finditer(solution))\n    if markers:\n        terminal_marker = TERMINAL_MARKER_RE.search(solution)\n        if len(markers) > 1 or terminal_marker is None:\n            flags.append(\'nonterminal_or_multiple_final_markers\')\n        elif answers_equivalent(terminal_marker.group(2), final_answer) is not True:\n            flags.append(\'unverified_terminal_marker_target\')\n    return list(dict.fromkeys(flags))\n\n\ndef build_completion(solution, final_answer):\n    if not isinstance(solution, str) or not solution.strip():\n        raise ValueError(\'A nonempty worked solution is required\')\n    # Preserve solution text and an already-correct terminal block; do not erase evidence.\n    final_answer = compact_answer(final_answer)\n    if answer_type(final_answer) == \'missing\' or answer_type(final_answer).startswith(\'unsupported\'):\n        raise ValueError(\'Missing or unsupported final target requires quarantine\')\n    markers = list(FINAL_MARKER_RE.finditer(solution))\n    marker = TERMINAL_MARKER_RE.search(solution)\n    if len(markers) > 1 or (markers and marker is None):\n        raise ValueError(\'Nonterminal or multiple final-answer markers require review\')\n    if marker:\n        if answers_equivalent(marker.group(2), final_answer) is not True:\n            raise ValueError(\'Existing final marker conflicts with or is unverified against target\')\n        if marker.group(1).casefold().startswith(\'final answer\') and \':\' in marker.group(1):\n            return solution\n        raise ValueError(\'Noncanonical final marker requires review before formatting\')\n    return f\'{solution.rstrip()}\\n\\nFinal answer: {final_answer}\'\n\n\nREVIEW_METADATA_COLUMNS = [\'reviewer\', \'reviewed_utc\', \'policy_version\', \'resolution\']\n\n\ndef save_review_sheet(frame, path):\n    """Preserve decisions on identical inputs; reject stale evidence rather than overwrite it."""\n    import pandas as pd\n    frame = frame.copy()\n    for col in REVIEW_METADATA_COLUMNS:\n        if col not in frame: frame[col] = \'\'\n    path = Path(path)\n    if path.exists():\n        previous = pd.read_csv(path, keep_default_na=False, dtype=str)\n        data_cols = [c for c in frame if not c.startswith(\'manual_\') and c not in REVIEW_METADATA_COLUMNS]\n        expected = frame[data_cols].fillna(\'\').astype(str).reset_index(drop=True)\n        if not set(data_cols).issubset(previous) or not previous[data_cols].reset_index(drop=True).equals(expected):\n            raise ValueError(f\'Stale review inputs: {path}; start a new run, do not overwrite review evidence\')\n        return previous\n    frame.to_csv(path, index=False)\n    return frame\n\n\ndef review_evidence(path, policy_version):\n    import pandas as pd\n    sheet = pd.read_csv(path, keep_default_na=False, dtype=str)\n    if sheet.empty:\n        if \'manual_same_problem_yes_no\' not in sheet:\n            raise ValueError(f\'Mandatory source/target review is empty: {path}\')\n        return file_sha256(path)  # no generated pairs; approval still binds this file\n    required = [c for c in sheet if c.startswith(\'manual_\') and\n                (c.endswith(\'_yes_no\') or c == \'manual_keep_quarantine\')]\n    if not required:\n        raise ValueError(f\'No structured review decisions: {path}\')\n    for col in REVIEW_METADATA_COLUMNS:\n        if col not in sheet or sheet[col].str.strip().eq(\'\').any():\n            raise ValueError(f\'Incomplete review metadata {col}: {path}\')\n    if not sheet[\'policy_version\'].eq(policy_version).all():\n        raise ValueError(f\'Wrong policy version: {path}\')\n    for value in sheet[\'reviewed_utc\']:\n        stamp = datetime.fromisoformat(value.replace(\'Z\', \'+00:00\'))\n        if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):\n            raise ValueError(f\'Invalid review date: {path}\')\n    for col in required:\n        allowed = {\'keep\', \'quarantine\'} if col == \'manual_keep_quarantine\' else {\'yes\', \'no\'}\n        if not sheet[col].str.casefold().isin(allowed).all():\n            raise ValueError(f\'Incomplete/invalid decision {col}: {path}\')\n    if not sheet[\'resolution\'].isin({\'accepted\', \'quarantined\', \'corrected_in_new_run\'}).all():\n        raise ValueError(f\'Unresolved review finding: {path}\')\n    # Accepted source/target examples cannot have failed checks. Pair \'no\' means distinct, valid.\n    if \'manual_same_problem_yes_no\' not in sheet:\n        accepted = sheet[\'resolution\'].eq(\'accepted\')\n        for col in required:\n            if col.endswith(\'_yes_no\') and col != \'manual_needs_missing_diagram_yes_no\':\n                if sheet.loc[accepted, col].str.casefold().ne(\'yes\').any():\n                    raise ValueError(f\'Failed target check accepted without repair: {path}\')\n        if \'manual_needs_missing_diagram_yes_no\' in sheet:\n            if sheet.loc[accepted, \'manual_needs_missing_diagram_yes_no\'].str.casefold().ne(\'no\').any():\n                raise ValueError(f\'Missing diagram accepted: {path}\')\n        if \'manual_keep_quarantine\' in sheet:\n            if sheet.loc[accepted, \'manual_keep_quarantine\'].str.casefold().ne(\'keep\').any():\n                raise ValueError(f\'Quarantine decision incorrectly accepted: {path}\')\n    return file_sha256(path)\n\n\ndef validate_review_approval(report_dir, filenames, policy, run_fingerprint, artifact_sha256=None):\n    approval_path = Path(report_dir) / \'review_approval.json\'\n    approval = json.loads(approval_path.read_text(encoding=\'utf-8\'))\n    if approval.get(\'policy\') != policy or approval.get(\'run_fingerprint\') != run_fingerprint:\n        raise ValueError(\'Review approval is for different policy/run inputs\')\n    if not approval.get(\'reviewer\') or not approval.get(\'reviewed_utc\'):\n        raise ValueError(\'Approval requires reviewer and date\')\n    stamp = datetime.fromisoformat(approval[\'reviewed_utc\'].replace(\'Z\', \'+00:00\'))\n    if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):\n        raise ValueError(\'Invalid approval date\')\n    if not approval.get(\'missed_match_audit\', {}).get(\'notes\'):\n        raise ValueError(\'Approval must document independently sought missed matches\')\n    evidence = {name: review_evidence(Path(report_dir) / name, policy[\'version\']) for name in filenames}\n    if approval.get(\'review_file_sha256\') != evidence:\n        raise ValueError(\'Review approval hashes do not match saved evidence\')\n    if artifact_sha256 is not None and approval.get(\'artifact_sha256\') != artifact_sha256:\n        raise ValueError(\'Review approval does not bind the current exported artifact hashes\')\n    return evidence\n\n\ndef validate_review_selection(report_dir, filenames, policy, selected_frames):\n    """Enforce decisions across training AND fresh pools, independent of similarity scores."""\n    import pandas as pd\n    columns = [\'row_id\', \'dataset_id\', \'original_row_id\']\n    if not selected_frames or any(not set(columns).issubset(frame) for frame in selected_frames):\n        raise ValueError(\'Review enforcement requires source identity for every selected pool\')\n    selected = pd.concat([frame[columns].astype(str) for frame in selected_frames], ignore_index=True)\n    selected_ids = set(selected[\'row_id\'])\n    source_ids = set(zip(selected[\'dataset_id\'], selected[\'original_row_id\']))\n    kinds = {\'duplicate_instance\', \'parameter_or_word_variant\', \'template_family\', \'unrelated\'}\n    for name in filenames:\n        sheet = pd.read_csv(Path(report_dir) / name, keep_default_na=False, dtype=str)\n        for row in sheet.to_dict(\'records\'):\n            resolution = row[\'resolution\']\n            if \'manual_same_problem_yes_no\' not in row:\n                excluded = resolution != \'accepted\' or row.get(\'manual_keep_quarantine\') == \'quarantine\'\n                if excluded and row.get(\'row_id\') in selected_ids:\n                    raise ValueError(f\'Review finding still selected: {name}/{row["row_id"]}\')\n                if excluded and (row.get(\'dataset\'), row.get(\'original_index\')) in source_ids:\n                    raise ValueError(f\'Source inspection finding still selected: {name}\')\n                continue\n            kind = row.get(\'manual_overlap_kind\')\n            same = row[\'manual_same_problem_yes_no\'].casefold()\n            action = row.get(\'manual_policy_action\', \'\').casefold()\n            if kind not in kinds or action not in {\'keep\', \'remove\'}:\n                raise ValueError(f\'Pair review requires classification and explicit keep/remove action: {name}\')\n            if (same == \'yes\') != (kind == \'duplicate_instance\'):\n                raise ValueError(f\'Contradictory same-instance classification: {name}\')\n            must_remove = kind == \'duplicate_instance\' or resolution != \'accepted\'\n            if kind == \'parameter_or_word_variant\' and policy[\'overlap_scope\'] == \'conservative_lexical_variants\':\n                must_remove = True\n            if must_remove and action != \'remove\':\n                raise ValueError(f\'Policy/review finding requires removal: {name}\')\n            if action == \'keep\':\n                continue\n            protected = \'training_row_id\' in row or row.get(\'search_method\', \'\').startswith(\'protected_\')\n            if protected:\n                training_id = row.get(\'training_row_id\', row.get(\'row_id_b\'))\n                retained = training_id in selected_ids\n            else:\n                retained = row.get(\'row_id_a\') in selected_ids and row.get(\'row_id_b\') in selected_ids\n            if retained:\n                raise ValueError(f\'Reviewed overlap remains selected despite removal decision: {name}\')\n\n\ndef verify_export_hashes(manifest_dir, report_dir, run_tag, summaries, prompt_spec_hash,\n                         fresh_summary=None, debug_summary=None):\n    """Rehash release artifacts; changing a file after export invalidates the gate."""\n    expected = {f\'train_n{size}_{run_tag}.jsonl\': summary[\'sha256\'] for size, summary in summaries.items()}\n    if fresh_summary is not None:\n        expected[f\'fresh_candidate_pool_R1000_{run_tag}.jsonl\'] = fresh_summary[\'sha256\']\n    if debug_summary is not None:\n        expected[f\'debug_32_{run_tag}.jsonl\'] = debug_summary[\'sha256\']\n    expected[\'prompt_specification.json\'] = prompt_spec_hash\n    for name, digest in expected.items():\n        path = Path(report_dir if name == \'prompt_specification.json\' else manifest_dir) / name\n        if file_sha256(path) != digest:\n            raise ValueError(f\'Exported artifact changed after generation: {name}\')\n    return expected\n\n\ndef reconcile_training_tokens(expected, verified):\n    """Require the exact actual training-manifest coverage before computing a budget."""\n    import pandas as pd\n    for frame, cols, name in [(expected, {\'row_id\', \'supervised_tokens\'}, \'expected\'),\n                              (verified, {\'row_id\', \'actual_supervised_tokens\'}, \'verified\')]:\n        if not cols.issubset(frame.columns) or frame.empty:\n            raise ValueError(f\'{name} token records are empty or missing columns\')\n        if frame[\'row_id\'].isna().any() or frame[\'row_id\'].astype(str).str.strip().eq(\'\').any():\n            raise ValueError(f\'{name} has missing IDs\')\n        if not frame[\'row_id\'].is_unique:\n            raise ValueError(f\'{name} has duplicate IDs\')\n    if set(expected[\'row_id\']) != set(verified[\'row_id\']):\n        raise ValueError(\'Training token coverage mismatch: missing or unexpected IDs\')\n    result = expected[[\'row_id\', \'supervised_tokens\']].merge(\n        verified[[\'row_id\', \'actual_supervised_tokens\']], on=\'row_id\', how=\'outer\', validate=\'one_to_one\', indicator=True)\n    values = pd.to_numeric(result[\'actual_supervised_tokens\'], errors=\'raise\')\n    if values.isna().any() or (~values.map(lambda n: n > 0 and n == int(n))).any():\n        raise ValueError(\'Actual label counts must be positive integers\')\n    result[\'actual_supervised_tokens\'] = values.astype(\'int64\')\n    result[\'difference\'] = result[\'actual_supervised_tokens\'] - result[\'supervised_tokens\']\n    if result[\'difference\'].ne(0).any():\n        raise ValueError(\'Actual training labels disagree with preprocessing counts\')\n    return result.drop(columns=\'_merge\')\n\n\ndef validate_problem_manifest(frame, label):\n    required = {\'set_id\', \'problem_id\', \'problem\'}\n    if not required.issubset(frame) or frame.empty:\n        raise ValueError(f\'{label} must contain nonempty set_id/problem_id/problem records\')\n    for col in required:\n        if frame[col].isna().any() or frame[col].astype(str).str.strip().eq(\'\').any():\n            raise ValueError(f\'{label} has missing {col}\')\n    if frame.duplicated([\'set_id\', \'problem_id\']).any():\n        raise ValueError(f\'{label} has duplicate composite problem IDs\')\n\n\ndef validate_parent_evidence(seen, actual_manifest, metadata, expected_manifest_sha256=None):\n    if seen.empty or \'problem_group_id\' not in seen or not seen[\'problem_group_id\'].is_unique:\n        raise ValueError(\'Parent-seen groups must be nonempty and unique\')\n    if seen[\'problem_group_id\'].isna().any() or seen[\'problem_group_id\'].astype(str).str.strip().eq(\'\').any():\n        raise ValueError(\'Parent-seen groups contain missing IDs\')\n    required = {\'checkpoint_id\', \'checkpoint_sha256\', \'actual_manifest_sha256\', \'confirmed_by\', \'confirmed_utc\'}\n    if not required.issubset(metadata) or any(not metadata[k] for k in required):\n        raise ValueError(\'Parent checkpoint confirmation is incomplete\')\n    if not re.fullmatch(r\'[0-9a-f]{64}\', metadata[\'checkpoint_sha256\']):\n        raise ValueError(\'Parent checkpoint requires a SHA-256 identity\')\n    stamp = datetime.fromisoformat(metadata[\'confirmed_utc\'].replace(\'Z\', \'+00:00\'))\n    if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):\n        raise ValueError(\'Parent confirmation requires a valid timestamp\')\n    if metadata[\'actual_manifest_sha256\'] != file_sha256(actual_manifest):\n        raise ValueError(\'Actual parent training manifest hash mismatch\')\n    if expected_manifest_sha256 is not None and metadata[\'actual_manifest_sha256\'] != expected_manifest_sha256:\n        raise ValueError(\'Parent checkpoint was not trained on the agreed corrected R1000 manifest\')\n    records = [json.loads(line) for line in Path(actual_manifest).read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n    if not records or len({r[\'row_id\'] for r in records}) != len(records):\n        raise ValueError(\'Actual parent training manifest has empty/duplicate IDs\')\n    actual_groups = {r[\'problem_group_id\'] for r in records}\n    if set(seen[\'problem_group_id\']) != actual_groups:\n        raise ValueError(\'Parent groups do not cover actual training manifest\')\n    return records\n'
# BEGIN EMBEDDED PIPELINE CONTROLS
"""Versioned, offline-testable controls embedded in Person1_Full_Run_v2.ipynb.

These checks establish parsing/structural properties, not mathematical correctness.
"""
import ast
import hashlib
import json
import math
import re
import unicodedata
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from fractions import Fraction
from pathlib import Path

CONTROL_VERSION = 'person1-v3.0'
PROMPT_SPEC_VERSION = 'math-worked-solution-v2'
PROMPT_TEMPLATE = (
    'Question: {problem}\n'
    'Solve the problem step by step. End with: Final answer: <answer>.\n'
    'Solution:\n'
)
NUMBER_PATTERN = r'[+-]?(?:(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?|\.\d+)(?:\s*/\s*[+-]?(?:\d+(?:\.\d+)?|\.\d+))?%?'
NUMBER_RE = re.compile(NUMBER_PATTERN)
SUSPECT_ANSWER_RE = re.compile(r'\(,|,\)|,,|,=|^,|,$|^=|\(\)')
CJK_RE = re.compile(r'[\u4e00-\u9fff]')


def clean_text(value):
    if value is None or type(value).__name__ in {'NAType', 'NaTType'}:
        return ''
    if not isinstance(value, str):
        try:
            if math.isnan(value): return ''
        except (TypeError, ValueError): pass
    text = unicodedata.normalize('NFKC', str(value)).replace('\r\n', '\n').replace('\r', '\n')
    return re.sub(r'\n{3,}', '\n\n', re.sub(r'[ \t]+', ' ', text)).strip()


def raw_text(value):
    """Preserve original string bytes; represent missing scalar text as empty."""
    return value if isinstance(value, str) else clean_text(value)


def compact_answer(value):
    # Never silently truncate an answer (v1 truncated at 500 characters).
    text = clean_text(value)
    return text[:-1].rstrip() if text.endswith('.') else text


def normalize_problem_for_matching(value):
    return re.sub(r'\s+', ' ', clean_text(value).casefold()).strip()


def sha256_text(value):
    return hashlib.sha256(value.encode('utf-8')).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def squash_answer(value):
    return re.sub(r'\s+|\\left|\\right|\$|\\!|\\,|\\;|\\displaystyle', '', compact_answer(value).casefold())


def scalar_fraction(value):
    """Safe exact arithmetic only; no eval, symbolic parsing or approximate equality."""
    text = squash_answer(value)
    text = re.sub(r'^([a-z])=', '', text)
    text = re.sub(r'\\(?:dfrac|tfrac|frac)\{([+-]?\d+)\}\{([+-]?\d+)\}', r'(\1/\2)', text)
    # Only thousands separators are numeric commas; (1,2) is not scalar 12.
    comma_checked = re.sub(r'(?<!\d)\d{1,3}(?:,\d{3})+(?!\d)', lambda match: match.group().replace(',', ''), text)
    if ',' in comma_checked: return None
    text = comma_checked
    percent = text.endswith('%')
    if percent:
        text = text[:-1]
    if not text or len(text) > 100 or not re.fullmatch(r'[\d.+*/() -]+', text):
        return None
    def visit(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return Fraction(ast.get_source_segment(text, node))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            result = visit(node.operand)
            return result if isinstance(node.op, ast.UAdd) else -result
        if isinstance(node, ast.BinOp):
            left, right = visit(node.left), visit(node.right)
            if isinstance(node.op, ast.Add): return left + right
            if isinstance(node.op, ast.Sub): return left - right
            if isinstance(node.op, ast.Mult): return left * right
            if isinstance(node.op, ast.Div): return left / right
        raise ValueError('unsupported scalar')
    try:
        value = visit(ast.parse(text, mode='eval').body)
        return value / 100 if percent else value
    except (ValueError, SyntaxError, ZeroDivisionError, TypeError):
        return None


def normalize_number(value):
    fraction = scalar_fraction(value)
    if fraction is None:
        return value
    return str(fraction.numerator) if fraction.denominator == 1 else f'{fraction.numerator}/{fraction.denominator}'


def answer_key(value):
    fraction = scalar_fraction(value)
    return 'rational:' + str(fraction) if fraction is not None else 'text:' + squash_answer(value)


def answers_equivalent(left, right, shared_checker=None):
    """True/False for exact rational scalars; None means equivalence is unproven."""
    if squash_answer(left) == squash_answer(right):
        return True
    a, b = scalar_fraction(left), scalar_fraction(right)
    if a is not None and b is not None:
        return a == b
    if shared_checker is not None:
        result = shared_checker(left, right)
        if result not in (True, False, None):
            raise ValueError('Shared checker must return True, False or None')
        return result
    return None


def suspected_multi_part(problem):
    text = clean_text(problem).casefold()
    # Conservative scope: multi-part examples await a reviewed full answer tuple.
    return bool(re.search(
        r'\b(?:both|respectively)\b|\b(?:radius|diameter|length|width|area|perimeter|volume)'
        r'\s+(?:and|,)\s+(?:the\s+)?(?:radius|diameter|length|width|area|perimeter|volume)\b'
        r'|\bhow much did each\b|\b(?:find|calculate|determine|what are)\b[^?\n]*\band\b'
        r'|(?:\(a\)|\(b\)|\bi\)|\bii\))', text))


def boxed_spans(solution):
    spans = []
    for marker in re.finditer(r'\\(?:boxed|fbox)\s*\{', solution):
        brace, depth = marker.end() - 1, 0
        for pos in range(brace, len(solution)):
            if solution[pos] == '{': depth += 1
            elif solution[pos] == '}':
                depth -= 1
                if depth == 0:
                    spans.append((marker.start(), pos + 1, compact_answer(solution[brace + 1:pos])))
                    break
    return spans


def last_boxed(solution):
    spans = boxed_spans(solution or '')
    return spans[-1][2] if spans else ''


FINAL_MARKER_RE = re.compile(r'(?im)^[ \t]*final answer[ \t]*[:=]')
TERMINAL_MARKER_RE = re.compile(r'(?im)^[ \t]*(final answer[ \t]*[:=]|####)[ \t]*([^\n]+?)[ \t]*(?:\n[ \t]*)*\Z')
MISSING_ANSWER_VALUES = {'', 'proof', 'notfound', 'nan', 'none', 'null', '<na>', 'nat', 'n/a'}


def answer_type(value):
    """Bounded shape validation; supported symbolic syntax is not a correctness proof."""
    text = compact_answer(value)
    if text.casefold() in MISSING_ANSWER_VALUES: return 'missing'
    if len(text) > 2000: return 'unsupported_oversized'
    if re.search(r'\b(?:or|and|respectively)\b|\\pm|±', text, re.I): return 'unsupported_multi_answer'
    without_numeric_commas = re.sub(r'\d{1,3}(?:,\d{3})+', '', text)
    if ',' in without_numeric_commas or ';' in text: return 'unsupported_multi_answer'
    if scalar_fraction(text) is not None: return 'numeric_scalar'
    # An equality chain must be reduced/reviewed explicitly, never accepted as its first number.
    if '=' in text and not re.fullmatch(r'[a-zA-Z]\s*=\s*[^=]+', text):
        return 'unsupported_equality_chain'
    if re.fullmatch(r'\$?' + NUMBER_PATTERN + r'\s+[A-Za-z][A-Za-z²³^ /-]*', text) and not re.search(
            r'\b(?:times|plus|minus|divided|pi|sqrt)\b', text, re.I):
        return 'quantity_scalar'
    core = text.strip('$')
    for left, right in [(r'\(', r'\)'), (r'\[', r'\]')]:
        if core.startswith(left) and core.endswith(right): core = core[len(left):-len(right)]
    stack = []
    for char in core:
        if char in '({[': stack.append(char)
        elif char in ')}]':
            if not stack or stack.pop() != {')':'(', '}':'{', ']':'['}[char]: return 'unsupported_unbalanced'
    if stack: return 'unsupported_unbalanced'
    commands = re.findall(r'\\([A-Za-z]+)', core)
    allowed = {'frac','dfrac','tfrac','sqrt','pi','alpha','beta','gamma','theta','lambda','sigma','omega',
               'infty','sin','cos','tan','log','ln','exp','left','right','cdot','times','boxed','fbox'}
    if any(command not in allowed for command in commands): return 'unsupported_symbolic_command'
    residue = re.sub(r'\\[A-Za-z]+', '', core)
    if not re.fullmatch(r'[A-Za-z0-9\s.+*/^_(){}\[\]=\-√π]+', residue): return 'unsupported_structure'
    identifiers = re.findall(r'[A-Za-z]+', residue)
    if any(len(name) > 1 and name not in {'sqrt','pi','sin','cos','tan','log','ln','exp','abs'} for name in identifiers):
        return 'unsupported_prose'
    if not commands and not identifiers and not re.search(r'[√π]', core): return 'unsupported_numeric_expression'
    return 'symbolic_scalar'


@dataclass(frozen=True)
class AnswerExtraction:
    candidate: str
    mode: str
    evidence_text: str = ''
    evidence_start: int = -1
    evidence_end: int = -1
    confidence: str = 'none'
    ambiguity_reason: str = ''
    answer_type: str = 'missing'
    evidence_basis: str = 'canonical_solution'

    def to_dict(self): return asdict(self)


def extract_orca_final_answer(problem, solution):
    """Only the terminal conclusion can supply a target. Never scan earlier givens."""
    solution = clean_text(solution)
    if not solution:
        return '', 'missing'
    if suspected_multi_part(problem):
        return '', 'ambiguous_multi_part'
    # Explicit markers must form the trailing block; earlier markers are ignored.
    markers = list(FINAL_MARKER_RE.finditer(solution))
    marker = TERMINAL_MARKER_RE.search(solution)
    if len(markers) > 1: return '', 'ambiguous_multiple_final_markers'
    if markers and marker is None: return '', 'ambiguous_nonterminal_final_marker'
    if marker:
        value = compact_answer(marker.group(2))
        kind = answer_type(value)
        if kind == 'missing':
            return '', 'missing'
        if kind.startswith('unsupported'): return '', 'ambiguous_' + kind
        return value, 'explicit_final_marker' if marker.group(1) != '####' else 'gsm_marker'
    boxes = boxed_spans(solution)
    if boxes:
        # A box inside the derivation is not a final answer; multiple boxes need review.
        if len(boxes) > 1:
            return '', 'ambiguous_multiple_boxes'
        _, end, value = boxes[-1]
        trailing = solution[end:]
        if re.fullmatch(r'[\s.$\\\[\]()!]*', trailing):
            return (value, 'boxed') if not answer_type(value).startswith('unsupported') else ('', 'ambiguous_boxed_structure')
    paragraph = solution.split('\n\n')[-1]
    sentence = re.split(r'(?<=[.!?])\s+(?=[A-Z])', paragraph)[-1].strip()
    # Anchored answer clause, e.g. '6 is subtracted ... result would be 6'.
    clause = re.search(
        r'(?i)\b(?:answer|result|number|value|total|cost|price|distance|area|radius|amount)'
        r'(?:\s+\w+){0,5}?\s+(?:is|equals|would be|will be)\s*[:=]?\s*\$?('
        + NUMBER_PATTERN + r')(?:\s+[A-Za-z][A-Za-z²³^ /-]*)?[.!]?$', sentence)
    if clause:
        # Numeric clauses with operator words require explicit review, not unit stripping.
        if re.search(r'\b(?:times|plus|minus|divided|pi|sqrt)\b', sentence[clause.start(1):], re.I):
            return '', 'ambiguous_unmarked_math_expression'
        return normalize_number(clause.group(1)), 'terminal_answer_clause'
    if re.search(r'\\(?:sqrt|frac|dfrac|tfrac|pi)\b|[√π]|[\w)]\s*\^|\b[a-zA-Z]\s*[+\-]\s*\d', sentence):
        return '', 'ambiguous_unmarked_math_expression'
    numbers = NUMBER_RE.findall(sentence)
    # Repeated numbers and arithmetic expressions remain ambiguous, never select 'novel' ones.
    if len(numbers) == 1:
        return normalize_number(numbers[0]), 'terminal_single_number_review'
    return '', 'ambiguous_terminal' if numbers else 'missing'


def extract_answer_metadata(problem, solution):
    text = clean_text(solution)
    answer, mode = extract_orca_final_answer(problem, text)
    marker = TERMINAL_MARKER_RE.search(text)
    boxes = boxed_spans(text)
    if marker:
        start, end = marker.span()
    elif mode == 'boxed' and boxes:
        start, end, _ = boxes[-1]
    else:
        start = text.rfind('\n\n') + 2 if '\n\n' in text else 0
        end = len(text)
    confidence = 'marked_terminal' if mode in {'explicit_final_marker','gsm_marker','boxed'} else (
        'terminal_clause' if mode == 'terminal_answer_clause' else 'heuristic' if answer else 'none')
    return AnswerExtraction(answer, mode, text[start:end], start, end, confidence,
                            mode.removeprefix('ambiguous_') if mode.startswith('ambiguous') else '', answer_type(answer))


def extraction_consistency_status(extraction, final_answer):
    if not extraction.candidate or extraction.confidence == 'heuristic':
        return 'unresolved_requires_review'
    equivalent = answers_equivalent(extraction.candidate, final_answer)
    return 'consistent_terminal_evidence' if equivalent is True else 'contradictory' if equivalent is False else 'unresolved_requires_review'


def diagram_flags(problem, solution):
    text = f'{problem}\n{solution}'.casefold()
    flags = []
    if re.search(r'\[/?asy\]|\\begin\{(?:asy|tikzpicture)\}|<svg\b', text):
        flags.append('inline_diagram_code')
    if re.search(r'!\[[^\]]*\]\s*(?:\(|\[)|<img\b|\\includegraphics\b'
                 r'|\b(?:figure|diagram)\s+(?:below|above)|shown in the (?:figure|diagram)', text):
        flags.append('missing_or_external_diagram')
    return flags


def has_image_reference(problem, solution):
    return bool(diagram_flags(problem, solution))


def looks_answer_only(solution, final_answer):
    text = clean_text(solution)
    if answers_equivalent(text, final_answer) is True:
        return True
    prose = re.sub(r'\\[a-zA-Z]+|[^a-zA-Z ]', ' ', text)
    words = prose.split()
    return len(words) < 8 and len(text.splitlines()) <= 2 and text.count('=') <= 1


def target_quality_flags(problem, solution, final_answer, answer_status):
    flags = []
    if answer_status.startswith('ambiguous'):
        flags.append('ambiguous_final_answer')
    kind = answer_type(final_answer)
    if kind == 'missing': flags.append('missing_or_sentinel_final_answer')
    elif kind.startswith('unsupported'): flags.append('unsupported_final_answer_structure')
    if answer_status == 'terminal_single_number_review':
        flags.append('heuristic_terminal_answer_requires_individual_review')
    if suspected_multi_part(problem):
        flags.append('suspected_multi_part_requires_review')
    if solution and looks_answer_only(solution, final_answer):
        flags.append('suspected_answer_only_solution')
    # A recognized terminal scalar disagreement is a separate consistency defect.
    terminal, status = extract_orca_final_answer(problem, solution)
    if terminal and status in {'explicit_final_marker', 'gsm_marker', 'boxed', 'terminal_answer_clause'}:
        if answers_equivalent(terminal, final_answer) is False:
            flags.append('terminal_answer_disagrees_with_target')
    markers = list(FINAL_MARKER_RE.finditer(solution))
    if markers:
        terminal_marker = TERMINAL_MARKER_RE.search(solution)
        if len(markers) > 1 or terminal_marker is None:
            flags.append('nonterminal_or_multiple_final_markers')
        elif answers_equivalent(terminal_marker.group(2), final_answer) is not True:
            flags.append('unverified_terminal_marker_target')
    return list(dict.fromkeys(flags))


def build_completion(solution, final_answer):
    if not isinstance(solution, str) or not solution.strip():
        raise ValueError('A nonempty worked solution is required')
    # Preserve solution text and an already-correct terminal block; do not erase evidence.
    final_answer = compact_answer(final_answer)
    if answer_type(final_answer) == 'missing' or answer_type(final_answer).startswith('unsupported'):
        raise ValueError('Missing or unsupported final target requires quarantine')
    markers = list(FINAL_MARKER_RE.finditer(solution))
    marker = TERMINAL_MARKER_RE.search(solution)
    if len(markers) > 1 or (markers and marker is None):
        raise ValueError('Nonterminal or multiple final-answer markers require review')
    if marker:
        if answers_equivalent(marker.group(2), final_answer) is not True:
            raise ValueError('Existing final marker conflicts with or is unverified against target')
        if marker.group(1).casefold().startswith('final answer') and ':' in marker.group(1):
            return solution
        raise ValueError('Noncanonical final marker requires review before formatting')
    return f'{solution.rstrip()}\n\nFinal answer: {final_answer}'


REVIEW_METADATA_COLUMNS = ['reviewer', 'reviewed_utc', 'policy_version', 'resolution']


def save_review_sheet(frame, path):
    """Preserve decisions on identical inputs; reject stale evidence rather than overwrite it."""
    import pandas as pd
    frame = frame.copy()
    for col in REVIEW_METADATA_COLUMNS:
        if col not in frame: frame[col] = ''
    path = Path(path)
    if path.exists():
        previous = pd.read_csv(path, keep_default_na=False, dtype=str)
        data_cols = [c for c in frame if not c.startswith('manual_') and c not in REVIEW_METADATA_COLUMNS]
        expected = frame[data_cols].fillna('').astype(str).reset_index(drop=True)
        if not set(data_cols).issubset(previous) or not previous[data_cols].reset_index(drop=True).equals(expected):
            raise ValueError(f'Stale review inputs: {path}; start a new run, do not overwrite review evidence')
        return previous
    frame.to_csv(path, index=False)
    return frame


def review_evidence(path, policy_version):
    import pandas as pd
    sheet = pd.read_csv(path, keep_default_na=False, dtype=str)
    if sheet.empty:
        if 'manual_same_problem_yes_no' not in sheet:
            raise ValueError(f'Mandatory source/target review is empty: {path}')
        return file_sha256(path)  # no generated pairs; approval still binds this file
    required = [c for c in sheet if c.startswith('manual_') and
                (c.endswith('_yes_no') or c == 'manual_keep_quarantine')]
    if not required:
        raise ValueError(f'No structured review decisions: {path}')
    for col in REVIEW_METADATA_COLUMNS:
        if col not in sheet or sheet[col].str.strip().eq('').any():
            raise ValueError(f'Incomplete review metadata {col}: {path}')
    if not sheet['policy_version'].eq(policy_version).all():
        raise ValueError(f'Wrong policy version: {path}')
    for value in sheet['reviewed_utc']:
        stamp = datetime.fromisoformat(value.replace('Z', '+00:00'))
        if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):
            raise ValueError(f'Invalid review date: {path}')
    for col in required:
        allowed = {'keep', 'quarantine'} if col == 'manual_keep_quarantine' else {'yes', 'no'}
        if not sheet[col].str.casefold().isin(allowed).all():
            raise ValueError(f'Incomplete/invalid decision {col}: {path}')
    if not sheet['resolution'].isin({'accepted', 'quarantined', 'corrected_in_new_run'}).all():
        raise ValueError(f'Unresolved review finding: {path}')
    # Accepted source/target examples cannot have failed checks. Pair 'no' means distinct, valid.
    if 'manual_same_problem_yes_no' not in sheet:
        accepted = sheet['resolution'].eq('accepted')
        for col in required:
            if col.endswith('_yes_no') and col != 'manual_needs_missing_diagram_yes_no':
                if sheet.loc[accepted, col].str.casefold().ne('yes').any():
                    raise ValueError(f'Failed target check accepted without repair: {path}')
        if 'manual_needs_missing_diagram_yes_no' in sheet:
            if sheet.loc[accepted, 'manual_needs_missing_diagram_yes_no'].str.casefold().ne('no').any():
                raise ValueError(f'Missing diagram accepted: {path}')
        if 'manual_keep_quarantine' in sheet:
            if sheet.loc[accepted, 'manual_keep_quarantine'].str.casefold().ne('keep').any():
                raise ValueError(f'Quarantine decision incorrectly accepted: {path}')
    return file_sha256(path)


def validate_review_approval(report_dir, filenames, policy, run_fingerprint, artifact_sha256=None):
    approval_path = Path(report_dir) / 'review_approval.json'
    approval = json.loads(approval_path.read_text(encoding='utf-8'))
    if approval.get('policy') != policy or approval.get('run_fingerprint') != run_fingerprint:
        raise ValueError('Review approval is for different policy/run inputs')
    if not approval.get('reviewer') or not approval.get('reviewed_utc'):
        raise ValueError('Approval requires reviewer and date')
    stamp = datetime.fromisoformat(approval['reviewed_utc'].replace('Z', '+00:00'))
    if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):
        raise ValueError('Invalid approval date')
    if not approval.get('missed_match_audit', {}).get('notes'):
        raise ValueError('Approval must document independently sought missed matches')
    evidence = {name: review_evidence(Path(report_dir) / name, policy['version']) for name in filenames}
    if approval.get('review_file_sha256') != evidence:
        raise ValueError('Review approval hashes do not match saved evidence')
    if artifact_sha256 is not None and approval.get('artifact_sha256') != artifact_sha256:
        raise ValueError('Review approval does not bind the current exported artifact hashes')
    return evidence


def validate_review_selection(report_dir, filenames, policy, selected_frames):
    """Enforce decisions across training AND fresh pools, independent of similarity scores."""
    import pandas as pd
    columns = ['row_id', 'dataset_id', 'original_row_id']
    if not selected_frames or any(not set(columns).issubset(frame) for frame in selected_frames):
        raise ValueError('Review enforcement requires source identity for every selected pool')
    selected = pd.concat([frame[columns].astype(str) for frame in selected_frames], ignore_index=True)
    selected_ids = set(selected['row_id'])
    source_ids = set(zip(selected['dataset_id'], selected['original_row_id']))
    kinds = {'duplicate_instance', 'parameter_or_word_variant', 'template_family', 'unrelated'}
    for name in filenames:
        sheet = pd.read_csv(Path(report_dir) / name, keep_default_na=False, dtype=str)
        for row in sheet.to_dict('records'):
            resolution = row['resolution']
            if 'manual_same_problem_yes_no' not in row:
                excluded = resolution != 'accepted' or row.get('manual_keep_quarantine') == 'quarantine'
                if excluded and row.get('row_id') in selected_ids:
                    raise ValueError(f'Review finding still selected: {name}/{row["row_id"]}')
                if excluded and (row.get('dataset'), row.get('original_index')) in source_ids:
                    raise ValueError(f'Source inspection finding still selected: {name}')
                continue
            kind = row.get('manual_overlap_kind')
            same = row['manual_same_problem_yes_no'].casefold()
            action = row.get('manual_policy_action', '').casefold()
            if kind not in kinds or action not in {'keep', 'remove'}:
                raise ValueError(f'Pair review requires classification and explicit keep/remove action: {name}')
            if (same == 'yes') != (kind == 'duplicate_instance'):
                raise ValueError(f'Contradictory same-instance classification: {name}')
            must_remove = kind == 'duplicate_instance' or resolution != 'accepted'
            if kind == 'parameter_or_word_variant' and policy['overlap_scope'] == 'conservative_lexical_variants':
                must_remove = True
            if must_remove and action != 'remove':
                raise ValueError(f'Policy/review finding requires removal: {name}')
            if action == 'keep':
                continue
            protected = 'training_row_id' in row or row.get('search_method', '').startswith('protected_')
            if protected:
                training_id = row.get('training_row_id', row.get('row_id_b'))
                retained = training_id in selected_ids
            else:
                retained = row.get('row_id_a') in selected_ids and row.get('row_id_b') in selected_ids
            if retained:
                raise ValueError(f'Reviewed overlap remains selected despite removal decision: {name}')


def verify_export_hashes(manifest_dir, report_dir, run_tag, summaries, prompt_spec_hash,
                         fresh_summary=None, debug_summary=None):
    """Rehash release artifacts; changing a file after export invalidates the gate."""
    expected = {f'train_n{size}_{run_tag}.jsonl': summary['sha256'] for size, summary in summaries.items()}
    if fresh_summary is not None:
        expected[f'fresh_candidate_pool_R1000_{run_tag}.jsonl'] = fresh_summary['sha256']
    if debug_summary is not None:
        expected[f'debug_32_{run_tag}.jsonl'] = debug_summary['sha256']
    expected['prompt_specification.json'] = prompt_spec_hash
    for name, digest in expected.items():
        path = Path(report_dir if name == 'prompt_specification.json' else manifest_dir) / name
        if file_sha256(path) != digest:
            raise ValueError(f'Exported artifact changed after generation: {name}')
    return expected


def reconcile_training_tokens(expected, verified):
    """Require the exact actual training-manifest coverage before computing a budget."""
    import pandas as pd
    for frame, cols, name in [(expected, {'row_id', 'supervised_tokens'}, 'expected'),
                              (verified, {'row_id', 'actual_supervised_tokens'}, 'verified')]:
        if not cols.issubset(frame.columns) or frame.empty:
            raise ValueError(f'{name} token records are empty or missing columns')
        if frame['row_id'].isna().any() or frame['row_id'].astype(str).str.strip().eq('').any():
            raise ValueError(f'{name} has missing IDs')
        if not frame['row_id'].is_unique:
            raise ValueError(f'{name} has duplicate IDs')
    if set(expected['row_id']) != set(verified['row_id']):
        raise ValueError('Training token coverage mismatch: missing or unexpected IDs')
    result = expected[['row_id', 'supervised_tokens']].merge(
        verified[['row_id', 'actual_supervised_tokens']], on='row_id', how='outer', validate='one_to_one', indicator=True)
    values = pd.to_numeric(result['actual_supervised_tokens'], errors='raise')
    if values.isna().any() or (~values.map(lambda n: n > 0 and n == int(n))).any():
        raise ValueError('Actual label counts must be positive integers')
    result['actual_supervised_tokens'] = values.astype('int64')
    result['difference'] = result['actual_supervised_tokens'] - result['supervised_tokens']
    if result['difference'].ne(0).any():
        raise ValueError('Actual training labels disagree with preprocessing counts')
    return result.drop(columns='_merge')


def validate_problem_manifest(frame, label):
    required = {'set_id', 'problem_id', 'problem'}
    if not required.issubset(frame) or frame.empty:
        raise ValueError(f'{label} must contain nonempty set_id/problem_id/problem records')
    for col in required:
        if frame[col].isna().any() or frame[col].astype(str).str.strip().eq('').any():
            raise ValueError(f'{label} has missing {col}')
    if frame.duplicated(['set_id', 'problem_id']).any():
        raise ValueError(f'{label} has duplicate composite problem IDs')


def validate_parent_evidence(seen, actual_manifest, metadata, expected_manifest_sha256=None):
    if seen.empty or 'problem_group_id' not in seen or not seen['problem_group_id'].is_unique:
        raise ValueError('Parent-seen groups must be nonempty and unique')
    if seen['problem_group_id'].isna().any() or seen['problem_group_id'].astype(str).str.strip().eq('').any():
        raise ValueError('Parent-seen groups contain missing IDs')
    required = {'checkpoint_id', 'checkpoint_sha256', 'actual_manifest_sha256', 'confirmed_by', 'confirmed_utc'}
    if not required.issubset(metadata) or any(not metadata[k] for k in required):
        raise ValueError('Parent checkpoint confirmation is incomplete')
    if not re.fullmatch(r'[0-9a-f]{64}', metadata['checkpoint_sha256']):
        raise ValueError('Parent checkpoint requires a SHA-256 identity')
    stamp = datetime.fromisoformat(metadata['confirmed_utc'].replace('Z', '+00:00'))
    if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):
        raise ValueError('Parent confirmation requires a valid timestamp')
    if metadata['actual_manifest_sha256'] != file_sha256(actual_manifest):
        raise ValueError('Actual parent training manifest hash mismatch')
    if expected_manifest_sha256 is not None and metadata['actual_manifest_sha256'] != expected_manifest_sha256:
        raise ValueError('Parent checkpoint was not trained on the agreed corrected R1000 manifest')
    records = [json.loads(line) for line in Path(actual_manifest).read_text(encoding='utf-8').splitlines() if line.strip()]
    if not records or len({r['row_id'] for r in records}) != len(records):
        raise ValueError('Actual parent training manifest has empty/duplicate IDs')
    actual_groups = {r['problem_group_id'] for r in records}
    if set(seen['problem_group_id']) != actual_groups:
        raise ValueError('Parent groups do not cover actual training manifest')
    return records

# END EMBEDDED PIPELINE CONTROLS







# BEGIN SHARED DATA CONTRACT
CONTRACT_SOURCE = '"""Shared plain-text data contract for Person 2/3/4; never loads model weights.\n\nAnswer consistency is a structural check, not a proof of mathematical correctness.\nThe notebook embeds this same source after the controls source.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nif \'build_completion\' not in globals():\n    from capstone_person1.pipeline_controls import (\n        PROMPT_TEMPLATE, build_completion, extract_answer_metadata, answers_equivalent,\n    )\n\nCONTRACT_VERSION = \'person1-plain-text-labels-v1\'\nTOKENIZER_ID = \'Qwen/Qwen2.5-0.5B\'\nPINNED_TOKENIZER_REVISION = \'060db6499f32faf8b98477b0a26969ef7d8b9987\'\nEXPECTED_END_TOKEN = \'<|endoftext|>\'\nEXPECTED_END_ID = 151643\n\n\nclass BoundaryError(ValueError):\n    """A token straddles prompt and completion; do not guess a loss mask."""\n\n\nclass ContextLengthError(ValueError):\n    """Exclude the whole row instead of truncating its reasoning or answer."""\n\n\ndef render_prompt(problem):\n    if not isinstance(problem, str) or not problem.strip():\n        raise ValueError(\'A nonempty problem string is required\')\n    return PROMPT_TEMPLATE.format(problem=problem)\n\n\ndef render_completion(solution, verified_final_answer):\n    evidence = extract_answer_metadata(\'\', solution)\n    if evidence.mode.startswith(\'ambiguous\'):\n        raise ValueError(\'Ambiguous solution conclusion requires review: \' + evidence.mode)\n    if not evidence.candidate:\n        raise ValueError(\'No supported terminal answer evidence; quarantine for review\')\n    if answers_equivalent(evidence.candidate, verified_final_answer) is not True:\n        raise ValueError(\'Target does not demonstrably agree with the terminal solution answer\')\n    if evidence.confidence == \'heuristic\':\n        raise ValueError(\'Heuristic answer requires an individual corrected/reviewed terminal block\')\n    return build_completion(solution, verified_final_answer)\n\n\ndef verify_tokenizer(tokenizer):\n    if not getattr(tokenizer, \'is_fast\', False):\n        raise ValueError(\'The data contract requires a fast tokenizer with offsets\')\n    for field, expected in [(\'eos_token\', EXPECTED_END_TOKEN), (\'pad_token\', EXPECTED_END_TOKEN),\n                            (\'eos_token_id\', EXPECTED_END_ID), (\'pad_token_id\', EXPECTED_END_ID)]:\n        if getattr(tokenizer, field, None) != expected:\n            raise ValueError(f\'Pinned tokenizer {field} differs from the agreed contract\')\n    if tokenizer.convert_tokens_to_ids(EXPECTED_END_TOKEN) != EXPECTED_END_ID:\n        raise ValueError(\'EOS vocabulary mapping differs from the agreed contract\')\n    return {field: getattr(tokenizer, field) for field in\n            [\'eos_token\', \'pad_token\', \'eos_token_id\', \'pad_token_id\', \'is_fast\']}\n\n\ndef load_tokenizer(*, cache_dir=None, local_files_only=False):\n    from transformers import AutoTokenizer\n    tokenizer = AutoTokenizer.from_pretrained(\n        TOKENIZER_ID, revision=PINNED_TOKENIZER_REVISION, use_fast=True,\n        cache_dir=str(cache_dir) if cache_dir is not None else None,\n        local_files_only=local_files_only,\n    )\n    verify_tokenizer(tokenizer)\n    return tokenizer\n\n\ndef encode_text_pair(tokenizer, prompt, completion, *, max_total_tokens=2048):\n    """Encode combined text, append EOS, then derive every budget from actual labels."""\n    verify_tokenizer(tokenizer)\n    if not isinstance(prompt, str) or not prompt.endswith(\'Solution:\\n\'):\n        raise ValueError(\'Prompt must end with the exact Solution newline\')\n    if not isinstance(completion, str) or not completion.strip():\n        raise ValueError(\'Completion must be nonempty\')\n    encoded = tokenizer(prompt + completion, add_special_tokens=False, return_offsets_mapping=True)\n    ids = list(encoded[\'input_ids\'])\n    offsets = list(encoded[\'offset_mapping\'])\n    if len(ids) != len(offsets) or not ids:\n        raise BoundaryError(\'Missing combined-text offsets\')\n    boundary = len(prompt)\n    prompt_mask = []\n    for left, right in offsets:\n        if not (0 <= left < right <= len(prompt) + len(completion)):\n            raise BoundaryError(\'Unsupported empty/invalid tokenizer offset\')\n        if left < boundary < right:\n            raise BoundaryError(\'Token crosses the prompt/completion character boundary\')\n        prompt_mask.append(right <= boundary)\n    prompt_tokens = sum(prompt_mask)\n    if not 0 < prompt_tokens < len(ids) or prompt_mask != [True] * prompt_tokens + [False] * (len(ids) - prompt_tokens):\n        raise BoundaryError(\'Combined-text offsets do not define a contiguous completion\')\n    if tokenizer.eos_token_id in ids:\n        raise ValueError(\'Input text already contains an EOS control token\')\n    labels = [-100 if is_prompt else token for token, is_prompt in zip(ids, prompt_mask)]\n    completion_tokens = sum(label != -100 for label in labels)\n    ids.append(tokenizer.eos_token_id)\n    labels.append(tokenizer.eos_token_id)\n    if max_total_tokens is not None and len(ids) > max_total_tokens:\n        raise ContextLengthError(f\'Whole example needs {len(ids)} tokens (limit {max_total_tokens})\')\n    # Separate prompt encoding is only a diagnostic; it never supplies labels or IDs.\n    prefix = tokenizer(prompt, add_special_tokens=False)[\'input_ids\']\n    return {\n        \'input_ids\': ids, \'attention_mask\': [1] * len(ids), \'labels\': labels,\n        \'prompt_text\': prompt, \'completion\': completion,\n        \'completion_start\': prompt_tokens, \'prompt_tokens\': prompt_tokens,\n        \'completion_tokens\': completion_tokens, \'eos_tokens\': 1,\n        \'supervised_tokens\': sum(label != -100 for label in labels),\n        \'total_tokens\': len(ids), \'boundary_mismatch\': ids[:len(prefix)] != list(prefix),\n        \'boundary_crossing_token\': False,\n    }\n\n\ndef encode_example(tokenizer, problem, solution, verified_final_answer, *, max_total_tokens=2048):\n    evidence = extract_answer_metadata(problem, solution)\n    if evidence.mode.startswith(\'ambiguous\'):\n        raise ValueError(\'Problem/solution requires review: \' + evidence.mode)\n    return encode_text_pair(tokenizer, render_prompt(problem),\n                            render_completion(solution, verified_final_answer),\n                            max_total_tokens=max_total_tokens)\n\n\ndef pad_encoded(encoded, length, *, pad_token_id=EXPECTED_END_ID):\n    """Right pad by position; the real EOS remains supervised even when PAD == EOS."""\n    if length < len(encoded[\'input_ids\']):\n        raise ContextLengthError(\'Padding must never truncate a row\')\n    result = dict(encoded)\n    count = length - len(encoded[\'input_ids\'])\n    result[\'input_ids\'] = list(encoded[\'input_ids\']) + [pad_token_id] * count\n    result[\'attention_mask\'] = list(encoded[\'attention_mask\']) + [0] * count\n    result[\'labels\'] = list(encoded[\'labels\']) + [-100] * count\n    if sum(label != -100 for label in result[\'labels\']) != encoded[\'supervised_tokens\']:\n        raise ValueError(\'Padding changed the supervised-token budget\')\n    return result\n\n\ndef validate_manifest_record(record, tokenizer, *, max_total_tokens=2048):\n    for key in [\'row_id\', \'problem_group_id\']:\n        if not isinstance(record.get(key), str) or not record[key]:\n            raise ValueError(\'Manifest requires nonempty \' + key)\n    encoded = encode_example(tokenizer, record[\'problem\'], record[\'solution\'], record[\'final_answer\'],\n                             max_total_tokens=max_total_tokens)\n    for key in [\'prompt_text\', \'completion\', \'prompt_tokens\', \'completion_tokens\', \'supervised_tokens\', \'total_tokens\']:\n        if record.get(key) != encoded[key]:\n            raise ValueError(\'Manifest differs from the shared encoder: \' + key)\n    return encoded\n\n\ndef validate_manifest(path, release_metadata, tokenizer=None):\n    """Stream a JSONL manifest, re-encode each row and verify IDs/groups/hash/counts."""\n    if release_metadata.get(\'tokenizer_id\') != TOKENIZER_ID or release_metadata.get(\'tokenizer_revision\') != PINNED_TOKENIZER_REVISION:\n        raise ValueError(\'Manifest release uses a different tokenizer identity/revision\')\n    path = Path(path)\n    if release_metadata.get(\'manifest_sha256\') != hashlib.sha256(path.read_bytes()).hexdigest():\n        raise ValueError(\'Manifest content hash differs from release metadata\')\n    if tokenizer is None:\n        tokenizer = load_tokenizer(cache_dir=release_metadata.get(\'tokenizer_cache_dir\'))\n    ids, groups, supervised = set(), set(), 0\n    with path.open(encoding=\'utf-8\') as stream:\n        for line in stream:\n            record = json.loads(line)\n            encoded = validate_manifest_record(record, tokenizer, max_total_tokens=release_metadata.get(\'max_total_tokens\', 2048))\n            if record[\'row_id\'] in ids or record[\'problem_group_id\'] in groups:\n                raise ValueError(\'Duplicate row/group in manifest\')\n            ids.add(record[\'row_id\'])\n            groups.add(record[\'problem_group_id\'])\n            supervised += encoded[\'supervised_tokens\']\n    if not ids or len(ids) != release_metadata.get(\'rows\'):\n        raise ValueError(\'Manifest is empty or row count differs from release metadata\')\n    if supervised != release_metadata.get(\'supervised_tokens\'):\n        raise ValueError(\'Manifest label-derived budget differs from release metadata\')\n    return {\'rows\': len(ids), \'supervised_tokens\': supervised, \'status\': \'passed\'}\n\n\ndef formatting_contract(tokenizer, *, asset_directory=None, max_total_tokens=2048):\n    verified = verify_tokenizer(tokenizer)\n    hashes = {}\n    if asset_directory is not None:\n        for name in [\'tokenizer.json\', \'tokenizer_config.json\', \'vocab.json\', \'merges.txt\']:\n            path = Path(asset_directory) / name\n            if path.is_file(): hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()\n        if not {\'tokenizer.json\', \'tokenizer_config.json\'} <= hashes.keys():\n            raise ValueError(\'Pinned tokenizer assets are missing\')\n    return {\n        \'contract_version\': CONTRACT_VERSION, \'prompt_template\': PROMPT_TEMPLATE,\n        \'prompt_template_sha256\': hashlib.sha256(PROMPT_TEMPLATE.encode(\'utf-8\')).hexdigest(),\n        \'tokenizer_id\': TOKENIZER_ID, \'tokenizer_revision\': PINNED_TOKENIZER_REVISION,\n        \'tokenizer_class\': type(tokenizer).__name__, \'tokenizer_verified_at_load\': True,\n        **verified, \'tokenizer_asset_sha256\': hashes,\n        \'add_special_tokens\': False, \'apply_chat_template\': False,\n        \'encoding\': \'combined_prompt_and_completion_with_character_offsets\',\n        \'completion_policy\': \'preserve_correct_terminal_block; quarantine_unproven_consistency\',\n        \'append_eos_count\': 1, \'loss\': \'completion_and_eos_only\', \'ignore_label\': -100,\n        \'padding\': \'right_padding; attention_mask=0; labels=-100_by_position\',\n        \'budget\': \'sum(label != -100 for label in labels)\', \'max_total_tokens\': max_total_tokens,\n        \'evaluation_input\': \'render_prompt(problem); no_reference_answer_or_eos\',\n        \'other_model_tokenizers\': \'load_corresponding_pinned_tokenizer_and_verify_before_reusing_counts\',\n    }\n'
"""Shared plain-text data contract for Person 2/3/4; never loads model weights.

Answer consistency is a structural check, not a proof of mathematical correctness.
The notebook embeds this same source after the controls source.
"""
import hashlib
import json
from pathlib import Path

if 'build_completion' not in globals():
    from capstone_person1.pipeline_controls import (
        PROMPT_TEMPLATE, build_completion, extract_answer_metadata, answers_equivalent,
    )

CONTRACT_VERSION = 'person1-plain-text-labels-v1'
TOKENIZER_ID = 'Qwen/Qwen2.5-0.5B'
PINNED_TOKENIZER_REVISION = '060db6499f32faf8b98477b0a26969ef7d8b9987'
EXPECTED_END_TOKEN = '<|endoftext|>'
EXPECTED_END_ID = 151643


class BoundaryError(ValueError):
    """A token straddles prompt and completion; do not guess a loss mask."""


class ContextLengthError(ValueError):
    """Exclude the whole row instead of truncating its reasoning or answer."""


def render_prompt(problem):
    if not isinstance(problem, str) or not problem.strip():
        raise ValueError('A nonempty problem string is required')
    return PROMPT_TEMPLATE.format(problem=problem)


def render_completion(solution, verified_final_answer):
    evidence = extract_answer_metadata('', solution)
    if evidence.mode.startswith('ambiguous'):
        raise ValueError('Ambiguous solution conclusion requires review: ' + evidence.mode)
    if not evidence.candidate:
        raise ValueError('No supported terminal answer evidence; quarantine for review')
    if answers_equivalent(evidence.candidate, verified_final_answer) is not True:
        raise ValueError('Target does not demonstrably agree with the terminal solution answer')
    if evidence.confidence == 'heuristic':
        raise ValueError('Heuristic answer requires an individual corrected/reviewed terminal block')
    return build_completion(solution, verified_final_answer)


def verify_tokenizer(tokenizer):
    if not getattr(tokenizer, 'is_fast', False):
        raise ValueError('The data contract requires a fast tokenizer with offsets')
    for field, expected in [('eos_token', EXPECTED_END_TOKEN), ('pad_token', EXPECTED_END_TOKEN),
                            ('eos_token_id', EXPECTED_END_ID), ('pad_token_id', EXPECTED_END_ID)]:
        if getattr(tokenizer, field, None) != expected:
            raise ValueError(f'Pinned tokenizer {field} differs from the agreed contract')
    if tokenizer.convert_tokens_to_ids(EXPECTED_END_TOKEN) != EXPECTED_END_ID:
        raise ValueError('EOS vocabulary mapping differs from the agreed contract')
    return {field: getattr(tokenizer, field) for field in
            ['eos_token', 'pad_token', 'eos_token_id', 'pad_token_id', 'is_fast']}


def load_tokenizer(*, cache_dir=None, local_files_only=False):
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_ID, revision=PINNED_TOKENIZER_REVISION, use_fast=True,
        cache_dir=str(cache_dir) if cache_dir is not None else None,
        local_files_only=local_files_only,
    )
    verify_tokenizer(tokenizer)
    return tokenizer


def encode_text_pair(tokenizer, prompt, completion, *, max_total_tokens=2048):
    """Encode combined text, append EOS, then derive every budget from actual labels."""
    verify_tokenizer(tokenizer)
    if not isinstance(prompt, str) or not prompt.endswith('Solution:\n'):
        raise ValueError('Prompt must end with the exact Solution newline')
    if not isinstance(completion, str) or not completion.strip():
        raise ValueError('Completion must be nonempty')
    encoded = tokenizer(prompt + completion, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(encoded['input_ids'])
    offsets = list(encoded['offset_mapping'])
    if len(ids) != len(offsets) or not ids:
        raise BoundaryError('Missing combined-text offsets')
    boundary = len(prompt)
    prompt_mask = []
    for left, right in offsets:
        if not (0 <= left < right <= len(prompt) + len(completion)):
            raise BoundaryError('Unsupported empty/invalid tokenizer offset')
        if left < boundary < right:
            raise BoundaryError('Token crosses the prompt/completion character boundary')
        prompt_mask.append(right <= boundary)
    prompt_tokens = sum(prompt_mask)
    if not 0 < prompt_tokens < len(ids) or prompt_mask != [True] * prompt_tokens + [False] * (len(ids) - prompt_tokens):
        raise BoundaryError('Combined-text offsets do not define a contiguous completion')
    if tokenizer.eos_token_id in ids:
        raise ValueError('Input text already contains an EOS control token')
    labels = [-100 if is_prompt else token for token, is_prompt in zip(ids, prompt_mask)]
    completion_tokens = sum(label != -100 for label in labels)
    ids.append(tokenizer.eos_token_id)
    labels.append(tokenizer.eos_token_id)
    if max_total_tokens is not None and len(ids) > max_total_tokens:
        raise ContextLengthError(f'Whole example needs {len(ids)} tokens (limit {max_total_tokens})')
    # Separate prompt encoding is only a diagnostic; it never supplies labels or IDs.
    prefix = tokenizer(prompt, add_special_tokens=False)['input_ids']
    return {
        'input_ids': ids, 'attention_mask': [1] * len(ids), 'labels': labels,
        'prompt_text': prompt, 'completion': completion,
        'completion_start': prompt_tokens, 'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens, 'eos_tokens': 1,
        'supervised_tokens': sum(label != -100 for label in labels),
        'total_tokens': len(ids), 'boundary_mismatch': ids[:len(prefix)] != list(prefix),
        'boundary_crossing_token': False,
    }


def encode_example(tokenizer, problem, solution, verified_final_answer, *, max_total_tokens=2048):
    evidence = extract_answer_metadata(problem, solution)
    if evidence.mode.startswith('ambiguous'):
        raise ValueError('Problem/solution requires review: ' + evidence.mode)
    return encode_text_pair(tokenizer, render_prompt(problem),
                            render_completion(solution, verified_final_answer),
                            max_total_tokens=max_total_tokens)


def pad_encoded(encoded, length, *, pad_token_id=EXPECTED_END_ID):
    """Right pad by position; the real EOS remains supervised even when PAD == EOS."""
    if length < len(encoded['input_ids']):
        raise ContextLengthError('Padding must never truncate a row')
    result = dict(encoded)
    count = length - len(encoded['input_ids'])
    result['input_ids'] = list(encoded['input_ids']) + [pad_token_id] * count
    result['attention_mask'] = list(encoded['attention_mask']) + [0] * count
    result['labels'] = list(encoded['labels']) + [-100] * count
    if sum(label != -100 for label in result['labels']) != encoded['supervised_tokens']:
        raise ValueError('Padding changed the supervised-token budget')
    return result


def validate_manifest_record(record, tokenizer, *, max_total_tokens=2048):
    for key in ['row_id', 'problem_group_id']:
        if not isinstance(record.get(key), str) or not record[key]:
            raise ValueError('Manifest requires nonempty ' + key)
    encoded = encode_example(tokenizer, record['problem'], record['solution'], record['final_answer'],
                             max_total_tokens=max_total_tokens)
    for key in ['prompt_text', 'completion', 'prompt_tokens', 'completion_tokens', 'supervised_tokens', 'total_tokens']:
        if record.get(key) != encoded[key]:
            raise ValueError('Manifest differs from the shared encoder: ' + key)
    return encoded


def validate_manifest(path, release_metadata, tokenizer=None):
    """Stream a JSONL manifest, re-encode each row and verify IDs/groups/hash/counts."""
    if release_metadata.get('tokenizer_id') != TOKENIZER_ID or release_metadata.get('tokenizer_revision') != PINNED_TOKENIZER_REVISION:
        raise ValueError('Manifest release uses a different tokenizer identity/revision')
    path = Path(path)
    if release_metadata.get('manifest_sha256') != hashlib.sha256(path.read_bytes()).hexdigest():
        raise ValueError('Manifest content hash differs from release metadata')
    if tokenizer is None:
        tokenizer = load_tokenizer(cache_dir=release_metadata.get('tokenizer_cache_dir'))
    ids, groups, supervised = set(), set(), 0
    with path.open(encoding='utf-8') as stream:
        for line in stream:
            record = json.loads(line)
            encoded = validate_manifest_record(record, tokenizer, max_total_tokens=release_metadata.get('max_total_tokens', 2048))
            if record['row_id'] in ids or record['problem_group_id'] in groups:
                raise ValueError('Duplicate row/group in manifest')
            ids.add(record['row_id'])
            groups.add(record['problem_group_id'])
            supervised += encoded['supervised_tokens']
    if not ids or len(ids) != release_metadata.get('rows'):
        raise ValueError('Manifest is empty or row count differs from release metadata')
    if supervised != release_metadata.get('supervised_tokens'):
        raise ValueError('Manifest label-derived budget differs from release metadata')
    return {'rows': len(ids), 'supervised_tokens': supervised, 'status': 'passed'}


def formatting_contract(tokenizer, *, asset_directory=None, max_total_tokens=2048):
    verified = verify_tokenizer(tokenizer)
    hashes = {}
    if asset_directory is not None:
        for name in ['tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt']:
            path = Path(asset_directory) / name
            if path.is_file(): hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()
        if not {'tokenizer.json', 'tokenizer_config.json'} <= hashes.keys():
            raise ValueError('Pinned tokenizer assets are missing')
    return {
        'contract_version': CONTRACT_VERSION, 'prompt_template': PROMPT_TEMPLATE,
        'prompt_template_sha256': hashlib.sha256(PROMPT_TEMPLATE.encode('utf-8')).hexdigest(),
        'tokenizer_id': TOKENIZER_ID, 'tokenizer_revision': PINNED_TOKENIZER_REVISION,
        'tokenizer_class': type(tokenizer).__name__, 'tokenizer_verified_at_load': True,
        **verified, 'tokenizer_asset_sha256': hashes,
        'add_special_tokens': False, 'apply_chat_template': False,
        'encoding': 'combined_prompt_and_completion_with_character_offsets',
        'completion_policy': 'preserve_correct_terminal_block; quarantine_unproven_consistency',
        'append_eos_count': 1, 'loss': 'completion_and_eos_only', 'ignore_label': -100,
        'padding': 'right_padding; attention_mask=0; labels=-100_by_position',
        'budget': 'sum(label != -100 for label in labels)', 'max_total_tokens': max_total_tokens,
        'evaluation_input': 'render_prompt(problem); no_reference_answer_or_eos',
        'other_model_tokenizers': 'load_corresponding_pinned_tokenizer_and_verify_before_reusing_counts',
    }

# END SHARED DATA CONTRACT


# BEGIN BENCHMARK MANIFEST INTERFACE
BENCHMARK_SOURCE = '"""Import Person 3\'s question-only frozen manifests; never builds evaluation data."""\nimport json\nimport re\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\n\nif \'validate_problem_manifest\' not in globals():\n    from capstone_person1.pipeline_controls import (\n        file_sha256, normalize_problem_for_matching, sha256_text, validate_problem_manifest,\n    )\n\nBENCHMARK_MANIFEST_VERSION = \'person3-question-manifest-v1\'\nEVALUATION_ROLES = {\'development\', \'fast_development\', \'final_test\'}\nPROVENANCE_FIELDS = [\'benchmark_family\', \'role\', \'dataset_id\', \'revision\', \'split\', \'selection_policy\']\n\n\ndef _validate_confirmation(metadata):\n    if not str(metadata.get(\'confirmed_by\', \'\')).strip() or not metadata.get(\'confirmed_utc\'):\n        raise ValueError(\'Person 3 confirmation identity/time is required\')\n    stamp = datetime.fromisoformat(metadata[\'confirmed_utc\'].replace(\'Z\', \'+00:00\'))\n    if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):\n        raise ValueError(\'Confirmation requires a timezone-aware, nonfuture timestamp\')\n\n\ndef _legacy_reserved_sets(metadata):\n    """Explicitly map the known historical manifest; never infer roles from filenames."""\n    sets = {}\n    for set_id, key, family in [(\'gsm8k_test\', \'gsm8k\', \'gsm8k\'), (\'math500_test\', \'math500\', \'math\')]:\n        source = metadata[\'datasets\'][key]\n        if source[\'split\'] != \'test\': raise ValueError(\'Historical reserved source must use its test split\')\n        sets[set_id] = dict(source, benchmark_family=family, role=\'final_test\',\n                            selection_policy={\'method\': \'all_official_test_questions\'},\n                            origin=\'historical_reserved_question_manifest\')\n    return sets\n\n\ndef load_question_manifest(path, metadata_path, *, legacy_reserved=False):\n    path, metadata_path = Path(path), Path(metadata_path)\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    validate_problem_manifest(frame, \'Person 3 question manifest\')\n    metadata = json.loads(metadata_path.read_text(encoding=\'utf-8\'))\n    if metadata.get(\'manifest_sha256\') != file_sha256(path):\n        raise ValueError(\'Question manifest content differs from frozen metadata hash\')\n    if metadata.get(\'set_counts\') != frame[\'set_id\'].value_counts().to_dict():\n        raise ValueError(\'Question manifest set counts differ from frozen metadata\')\n    legacy = legacy_reserved and \'sets\' not in metadata\n    if legacy:\n        sets = _legacy_reserved_sets(metadata)\n        confirmed = False\n    else:\n        if metadata.get(\'schema_version\') != BENCHMARK_MANIFEST_VERSION:\n            raise ValueError(\'Question metadata requires the explicit Person 3 manifest schema\')\n        _validate_confirmation(metadata)\n        sets, confirmed = metadata.get(\'sets\', {}), True\n    if set(sets) != set(frame[\'set_id\']):\n        raise ValueError(\'Every set requires explicit role/source metadata; unknown sets are invalid\')\n    parts = []\n    for set_id, source in sets.items():\n        if any(key not in source or source[key] in (\'\', None) for key in PROVENANCE_FIELDS):\n            raise ValueError(\'Incomplete source/role/selection provenance for \' + set_id)\n        if source[\'role\'] not in EVALUATION_ROLES or source[\'benchmark_family\'] not in {\'gsm8k\', \'math\'}:\n            raise ValueError(\'Unsupported evaluation role/family for this experiment\')\n        if not re.fullmatch(r\'[0-9a-f]{40}|sha256:[0-9a-f]{64}\', source[\'revision\']):\n            raise ValueError(\'Evaluation source requires an immutable revision or source-file digest\')\n        selection = source[\'selection_policy\']\n        if not isinstance(selection, dict) or not str(selection.get(\'method\', \'\')).strip():\n            raise ValueError(\'Selection policy requires a documented method\')\n        if re.search(r\'random|sample\', str(selection[\'method\']), re.I) and type(selection.get(\'seed\')) is not int:\n            raise ValueError(\'Sampled evaluation questions require an explicit integer seed\')\n        if source[\'role\'] != \'final_test\' and (source[\'dataset_id\'] == \'HuggingFaceH4/MATH-500\' or\n                (source[\'dataset_id\'] == \'openai/gsm8k\' and source[\'split\'] == \'test\')):\n            raise ValueError(\'Official reserved final tests cannot supply development questions\')\n        part = frame.loc[frame[\'set_id\'].eq(set_id), [\'set_id\', \'problem_id\', \'problem\']].copy()\n        for key in [\'benchmark_family\', \'role\', \'dataset_id\', \'revision\', \'split\']:\n            part[key] = source[key]\n        part[\'configuration\'] = source.get(\'configuration\', \'\')\n        part[\'selection_policy_json\'] = json.dumps(selection, sort_keys=True)\n        part[\'manifest_origin\'] = source.get(\'origin\', \'person3_confirmed_manifest\')\n        part[\'parent_set_id\'] = source.get(\'parent_set_id\', \'\')\n        part[\'role_confirmed_by_person3\'] = confirmed\n        part[\'source_manifest_sha256\'] = metadata[\'manifest_sha256\']\n        parts.append(part)\n    output = pd.concat(parts, ignore_index=True)\n    output[\'problem_hash\'] = output[\'problem\'].map(normalize_problem_for_matching).map(sha256_text)\n    for set_id, rows in output.groupby(\'set_id\'):\n        if rows[\'problem_hash\'].duplicated().any():\n            raise ValueError(\'Duplicate normalized questions inside \' + set_id)\n    return output, metadata\n\n\ndef _validate_roles(frame):\n    sets = {set_id: rows for set_id, rows in frame.groupby(\'set_id\')}\n    dev = frame[frame[\'role\'].isin([\'development\', \'fast_development\'])]\n    final = frame[frame[\'role\'].eq(\'final_test\')]\n    if set(dev[\'problem_hash\']) & set(final[\'problem_hash\']):\n        raise ValueError(\'Development questions intersect reserved final tests\')\n    # Source IDs additionally catch changed wording under the same frozen source identity.\n    source_keys = [\'dataset_id\', \'revision\', \'split\', \'problem_id\']\n    if set(map(tuple, dev[source_keys].values)) & set(map(tuple, final[source_keys].values)):\n        raise ValueError(\'Development/final source question identities intersect\')\n    for set_id, rows in sets.items():\n        if rows.iloc[0][\'role\'] != \'fast_development\': continue\n        parent = rows.iloc[0][\'parent_set_id\']\n        if not parent or parent not in sets or sets[parent].iloc[0][\'role\'] != \'development\':\n            raise ValueError(\'Fast-development set requires a supplied full-development parent\')\n        full = sets[parent]\n        for field in [\'benchmark_family\', \'dataset_id\', \'revision\', \'configuration\', \'split\']:\n            if rows.iloc[0][field] != full.iloc[0][field]:\n                raise ValueError(\'Fast/full source provenance differs\')\n        full_ids = dict(zip(full[\'problem_id\'], full[\'problem_hash\']))\n        if any(full_ids.get(row.problem_id) != row.problem_hash for row in rows.itertuples()):\n            raise ValueError(\'Fast-development IDs/text must be an exact subset of its full parent\')\n\n\ndef load_protection_inputs(reserved_path, reserved_metadata_path, development_path=None, development_metadata_path=None):\n    reserved, reserved_metadata = load_question_manifest(reserved_path, reserved_metadata_path, legacy_reserved=True)\n    if not reserved[\'role\'].eq(\'final_test\').all():\n        raise ValueError(\'Reserved manifest may contain only explicit final-test sets\')\n    if set(reserved[\'benchmark_family\']) != {\'gsm8k\', \'math\'}:\n        raise ValueError(\'This experiment requires both configured final-test families\')\n    inputs = {\'reserved_test_sha256\': file_sha256(reserved_path),\n              \'reserved_metadata_sha256\': file_sha256(reserved_metadata_path)}\n    development, parts = None, [reserved]\n    if (development_path is None) != (development_metadata_path is None):\n        raise ValueError(\'Development manifest and metadata must be supplied together\')\n    if development_path is not None:\n        exists = [Path(development_path).exists(), Path(development_metadata_path).exists()]\n        if any(exists) and not all(exists):\n            raise ValueError(\'Incomplete development manifest/metadata pair must be fixed\')\n        if all(exists):\n            development, _ = load_question_manifest(development_path, development_metadata_path)\n            if development[\'role\'].eq(\'final_test\').any():\n                raise ValueError(\'Development input cannot redefine reserved final tests\')\n            if set(development[\'set_id\']) & set(reserved[\'set_id\']):\n                raise ValueError(\'Development and final tests require distinct set IDs\')\n            parts.append(development)\n            inputs.update(development_sha256=file_sha256(development_path),\n                          development_metadata_sha256=file_sha256(development_metadata_path))\n    combined = pd.concat(parts, ignore_index=True)\n    _validate_roles(combined)\n    full = combined[combined[\'role\'].eq(\'development\')]\n    fast = combined[combined[\'role\'].eq(\'fast_development\')]\n    development_complete = set(full[\'benchmark_family\']) == {\'gsm8k\', \'math\'}\n    fast_complete = set(fast[\'benchmark_family\']) == {\'gsm8k\', \'math\'}\n    report = {\n        \'schema_version\': BENCHMARK_MANIFEST_VERSION, \'question_manifests_validated\': True,\n        \'inputs\': inputs, \'set_counts\': combined[\'set_id\'].value_counts().to_dict(),\n        \'role_counts\': combined[\'role\'].value_counts().to_dict(),\n        \'protected_question_entries\': len(combined), \'unique_protected_question_texts\': combined[\'problem_hash\'].nunique(),\n        \'person3_development_coverage\': \'complete\' if development_complete else \'pending\',\n        \'person3_fast_development_coverage\': \'complete\' if fast_complete else \'pending\',\n        \'person3_final_test_role_confirmation\': \'confirmed\' if reserved[\'role_confirmed_by_person3\'].all() else \'pending\',\n        \'development_protected\': development_complete,\n        \'fast_subsets_validated\': bool(len(fast)),\n        \'overlap_checks_status\': \'pending_pipeline_execution\',\n        \'person1_data_work_may_continue\': True,\n        \'evaluation_data_created_by_person1\': False,\n        \'limitations\': [\'This validates frozen question inputs; it does not run generation/scoring or certify near-match recall.\',\n                        \'Fast development entries are nested duplicates of full development; do not add their counts as unique questions.\'],\n    }\n    return {\'reserved\': reserved, \'development\': development, \'questions\': combined,\n            \'inputs\': inputs, \'development_protected\': development_complete, \'scope\': report}\n\n\ndef export_protection_inputs(bundle, processed_dir, report_dir):\n    processed_dir, report_dir = Path(processed_dir), Path(report_dir)\n    processed_dir.mkdir(parents=True, exist_ok=True)\n    report_dir.mkdir(parents=True, exist_ok=True)\n    path = processed_dir / \'protected_questions.csv\'\n    csv_text = bundle[\'questions\'].to_csv(index=False, lineterminator=\'\\n\')\n    if path.exists() and path.read_text(encoding=\'utf-8\') != csv_text:\n        raise ValueError(\'Protected-question export changed; start a new immutable run\')\n    path.write_text(csv_text, encoding=\'utf-8\', newline=\'\')\n    report = dict(bundle[\'scope\'], protected_questions_sha256=file_sha256(path))\n    (report_dir / \'evaluation_protection_scope.json\').write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf-8\')\n    return report\n'
"""Import Person 3's question-only frozen manifests; never builds evaluation data."""
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

if 'validate_problem_manifest' not in globals():
    from capstone_person1.pipeline_controls import (
        file_sha256, normalize_problem_for_matching, sha256_text, validate_problem_manifest,
    )

BENCHMARK_MANIFEST_VERSION = 'person3-question-manifest-v1'
EVALUATION_ROLES = {'development', 'fast_development', 'final_test'}
PROVENANCE_FIELDS = ['benchmark_family', 'role', 'dataset_id', 'revision', 'split', 'selection_policy']


def _validate_confirmation(metadata):
    if not str(metadata.get('confirmed_by', '')).strip() or not metadata.get('confirmed_utc'):
        raise ValueError('Person 3 confirmation identity/time is required')
    stamp = datetime.fromisoformat(metadata['confirmed_utc'].replace('Z', '+00:00'))
    if stamp.tzinfo is None or stamp > datetime.now(timezone.utc):
        raise ValueError('Confirmation requires a timezone-aware, nonfuture timestamp')


def _legacy_reserved_sets(metadata):
    """Explicitly map the known historical manifest; never infer roles from filenames."""
    sets = {}
    for set_id, key, family in [('gsm8k_test', 'gsm8k', 'gsm8k'), ('math500_test', 'math500', 'math')]:
        source = metadata['datasets'][key]
        if source['split'] != 'test': raise ValueError('Historical reserved source must use its test split')
        sets[set_id] = dict(source, benchmark_family=family, role='final_test',
                            selection_policy={'method': 'all_official_test_questions'},
                            origin='historical_reserved_question_manifest')
    return sets


def load_question_manifest(path, metadata_path, *, legacy_reserved=False):
    path, metadata_path = Path(path), Path(metadata_path)
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    validate_problem_manifest(frame, 'Person 3 question manifest')
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    if metadata.get('manifest_sha256') != file_sha256(path):
        raise ValueError('Question manifest content differs from frozen metadata hash')
    if metadata.get('set_counts') != frame['set_id'].value_counts().to_dict():
        raise ValueError('Question manifest set counts differ from frozen metadata')
    legacy = legacy_reserved and 'sets' not in metadata
    if legacy:
        sets = _legacy_reserved_sets(metadata)
        confirmed = False
    else:
        if metadata.get('schema_version') != BENCHMARK_MANIFEST_VERSION:
            raise ValueError('Question metadata requires the explicit Person 3 manifest schema')
        _validate_confirmation(metadata)
        sets, confirmed = metadata.get('sets', {}), True
    if set(sets) != set(frame['set_id']):
        raise ValueError('Every set requires explicit role/source metadata; unknown sets are invalid')
    parts = []
    for set_id, source in sets.items():
        if any(key not in source or source[key] in ('', None) for key in PROVENANCE_FIELDS):
            raise ValueError('Incomplete source/role/selection provenance for ' + set_id)
        if source['role'] not in EVALUATION_ROLES or source['benchmark_family'] not in {'gsm8k', 'math'}:
            raise ValueError('Unsupported evaluation role/family for this experiment')
        if not re.fullmatch(r'[0-9a-f]{40}|sha256:[0-9a-f]{64}', source['revision']):
            raise ValueError('Evaluation source requires an immutable revision or source-file digest')
        selection = source['selection_policy']
        if not isinstance(selection, dict) or not str(selection.get('method', '')).strip():
            raise ValueError('Selection policy requires a documented method')
        if re.search(r'random|sample', str(selection['method']), re.I) and type(selection.get('seed')) is not int:
            raise ValueError('Sampled evaluation questions require an explicit integer seed')
        if source['role'] != 'final_test' and (source['dataset_id'] == 'HuggingFaceH4/MATH-500' or
                (source['dataset_id'] == 'openai/gsm8k' and source['split'] == 'test')):
            raise ValueError('Official reserved final tests cannot supply development questions')
        part = frame.loc[frame['set_id'].eq(set_id), ['set_id', 'problem_id', 'problem']].copy()
        for key in ['benchmark_family', 'role', 'dataset_id', 'revision', 'split']:
            part[key] = source[key]
        part['configuration'] = source.get('configuration', '')
        part['selection_policy_json'] = json.dumps(selection, sort_keys=True)
        part['manifest_origin'] = source.get('origin', 'person3_confirmed_manifest')
        part['parent_set_id'] = source.get('parent_set_id', '')
        part['role_confirmed_by_person3'] = confirmed
        part['source_manifest_sha256'] = metadata['manifest_sha256']
        parts.append(part)
    output = pd.concat(parts, ignore_index=True)
    output['problem_hash'] = output['problem'].map(normalize_problem_for_matching).map(sha256_text)
    for set_id, rows in output.groupby('set_id'):
        if rows['problem_hash'].duplicated().any():
            raise ValueError('Duplicate normalized questions inside ' + set_id)
    return output, metadata


def _validate_roles(frame):
    sets = {set_id: rows for set_id, rows in frame.groupby('set_id')}
    dev = frame[frame['role'].isin(['development', 'fast_development'])]
    final = frame[frame['role'].eq('final_test')]
    if set(dev['problem_hash']) & set(final['problem_hash']):
        raise ValueError('Development questions intersect reserved final tests')
    # Source IDs additionally catch changed wording under the same frozen source identity.
    source_keys = ['dataset_id', 'revision', 'split', 'problem_id']
    if set(map(tuple, dev[source_keys].values)) & set(map(tuple, final[source_keys].values)):
        raise ValueError('Development/final source question identities intersect')
    for set_id, rows in sets.items():
        if rows.iloc[0]['role'] != 'fast_development': continue
        parent = rows.iloc[0]['parent_set_id']
        if not parent or parent not in sets or sets[parent].iloc[0]['role'] != 'development':
            raise ValueError('Fast-development set requires a supplied full-development parent')
        full = sets[parent]
        for field in ['benchmark_family', 'dataset_id', 'revision', 'configuration', 'split']:
            if rows.iloc[0][field] != full.iloc[0][field]:
                raise ValueError('Fast/full source provenance differs')
        full_ids = dict(zip(full['problem_id'], full['problem_hash']))
        if any(full_ids.get(row.problem_id) != row.problem_hash for row in rows.itertuples()):
            raise ValueError('Fast-development IDs/text must be an exact subset of its full parent')


def load_protection_inputs(reserved_path, reserved_metadata_path, development_path=None, development_metadata_path=None):
    reserved, reserved_metadata = load_question_manifest(reserved_path, reserved_metadata_path, legacy_reserved=True)
    if not reserved['role'].eq('final_test').all():
        raise ValueError('Reserved manifest may contain only explicit final-test sets')
    if set(reserved['benchmark_family']) != {'gsm8k', 'math'}:
        raise ValueError('This experiment requires both configured final-test families')
    inputs = {'reserved_test_sha256': file_sha256(reserved_path),
              'reserved_metadata_sha256': file_sha256(reserved_metadata_path)}
    development, parts = None, [reserved]
    if (development_path is None) != (development_metadata_path is None):
        raise ValueError('Development manifest and metadata must be supplied together')
    if development_path is not None:
        exists = [Path(development_path).exists(), Path(development_metadata_path).exists()]
        if any(exists) and not all(exists):
            raise ValueError('Incomplete development manifest/metadata pair must be fixed')
        if all(exists):
            development, _ = load_question_manifest(development_path, development_metadata_path)
            if development['role'].eq('final_test').any():
                raise ValueError('Development input cannot redefine reserved final tests')
            if set(development['set_id']) & set(reserved['set_id']):
                raise ValueError('Development and final tests require distinct set IDs')
            parts.append(development)
            inputs.update(development_sha256=file_sha256(development_path),
                          development_metadata_sha256=file_sha256(development_metadata_path))
    combined = pd.concat(parts, ignore_index=True)
    _validate_roles(combined)
    full = combined[combined['role'].eq('development')]
    fast = combined[combined['role'].eq('fast_development')]
    development_complete = set(full['benchmark_family']) == {'gsm8k', 'math'}
    fast_complete = set(fast['benchmark_family']) == {'gsm8k', 'math'}
    report = {
        'schema_version': BENCHMARK_MANIFEST_VERSION, 'question_manifests_validated': True,
        'inputs': inputs, 'set_counts': combined['set_id'].value_counts().to_dict(),
        'role_counts': combined['role'].value_counts().to_dict(),
        'protected_question_entries': len(combined), 'unique_protected_question_texts': combined['problem_hash'].nunique(),
        'person3_development_coverage': 'complete' if development_complete else 'pending',
        'person3_fast_development_coverage': 'complete' if fast_complete else 'pending',
        'person3_final_test_role_confirmation': 'confirmed' if reserved['role_confirmed_by_person3'].all() else 'pending',
        'development_protected': development_complete,
        'fast_subsets_validated': bool(len(fast)),
        'overlap_checks_status': 'pending_pipeline_execution',
        'person1_data_work_may_continue': True,
        'evaluation_data_created_by_person1': False,
        'limitations': ['This validates frozen question inputs; it does not run generation/scoring or certify near-match recall.',
                        'Fast development entries are nested duplicates of full development; do not add their counts as unique questions.'],
    }
    return {'reserved': reserved, 'development': development, 'questions': combined,
            'inputs': inputs, 'development_protected': development_complete, 'scope': report}


def export_protection_inputs(bundle, processed_dir, report_dir):
    processed_dir, report_dir = Path(processed_dir), Path(report_dir)
    processed_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    path = processed_dir / 'protected_questions.csv'
    csv_text = bundle['questions'].to_csv(index=False, lineterminator='\n')
    if path.exists() and path.read_text(encoding='utf-8') != csv_text:
        raise ValueError('Protected-question export changed; start a new immutable run')
    path.write_text(csv_text, encoding='utf-8', newline='')
    report = dict(bundle['scope'], protected_questions_sha256=file_sha256(path))
    (report_dir / 'evaluation_protection_scope.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    return report

# END BENCHMARK MANIFEST INTERFACE

# Freeze the entire preprocessing configuration BEFORE creating run reports.
frozen_configuration = {
    'control_source_sha256': sha256_text(CONTROL_SOURCE),
    'structured_data_configuration': DATA_CONFIG,
    'reference_encoder_sha256': sha256_text(CONTRACT_SOURCE),
    'benchmark_manifest_source_sha256': sha256_text(BENCHMARK_SOURCE),
    'dataset_revisions': {'orca': ORCA_REVISION, 'numina': NUMINA_REVISION},
    'tokenizer_id': TOKENIZER_ID, 'tokenizer_revision': TOKENIZER_REVISION,
    'reserved_repository_revisions': {'gsm8k': GSM8K_REVISION, 'math500': MATH500_REVISION},
    'split_seed': SPLIT_SEED, 'selection_seed': SELECTION_SEED,
    'candidate_universe_per_source': CANDIDATE_UNIVERSE_PER_SOURCE,
    'lsh_candidate_threshold': LSH_CANDIDATE_THRESHOLD,
    'minhash_permutations': MINHASH_PERMUTATIONS,
    'max_total_tokens': MAX_TOTAL_TOKENS, 'template_sha256': sha256_text(PROMPT_TEMPLATE),
    'manifest_sizes': MANIFEST_SIZES, 'target_per_source': TARGET_PER_SOURCE, 'target_pool': TARGET_WORKING_POOL,
    'fresh_pool_target': FRESH_POOL_TARGET, 'run_mode': RUN_MODE,
    'pilot_rows_per_dataset': PILOT_ROWS_PER_DATASET,
    'exclude_synthetic_numina': EXCLUDE_SYNTHETIC_NUMINA,
    'review_policy': REVIEW_POLICY,
    'protection_inputs': {
        path.name: file_sha256(path) if path.exists() else None
        for path in [EVAL_MANIFEST_PATH, INPUT_DIR / 'reserved_evaluation_metadata.json', DEV_MANIFEST_PATH, DEV_METADATA_PATH]
    },
    'shared_equivalence_sha256': file_sha256(SHARED_EQUIVALENCE_PATH) if SHARED_EQUIVALENCE_PATH.exists() else None,
}
RUN_FINGERPRINT = sha256_text(json.dumps(frozen_configuration, sort_keys=True))
config_path = RUN_ROOT / 'run_configuration.json'
if config_path.exists() and json.loads(config_path.read_text(encoding='utf-8')) != frozen_configuration:
    raise ValueError('Frozen run inputs/configuration changed; start a new versioned run')
config_path.write_text(json.dumps(frozen_configuration, indent=2), encoding='utf-8')

PACKAGE_NAMES = ['datasets', 'transformers', 'datasketch', 'huggingface_hub', 'pandas', 'pyarrow', 'numpy']
environment = {
    'python': sys.version,
    'packages': {name: importlib_metadata.version(name) for name in PACKAGE_NAMES},
}

with open(REPORT_DIR / 'environment.json', 'w', encoding='utf-8') as f:
    json.dump(environment, f, indent=2)

DECISIONS_LOG = REPORT_DIR / 'decisions_log.md'
def log_decision(message):
    with open(DECISIONS_LOG, 'a', encoding='utf-8') as f:
        f.write(f'- {pd.Timestamp.now().isoformat()} | {message}\n')

log_decision(
    f'run_mode={RUN_MODE}; split_seed={SPLIT_SEED}; selection_seed={SELECTION_SEED}; '
    f'max_total_tokens={MAX_TOTAL_TOKENS}; exclude_synthetic_numina={EXCLUDE_SYNTHETIC_NUMINA}'
)

environment

5 Load configured immutable dataset revisions and load the data

In [ ]:
api = HfApi()
orca_revision = ORCA_REVISION
numina_revision = NUMINA_REVISION
tokenizer_revision = TOKENIZER_REVISION
for revision in [orca_revision, numina_revision, tokenizer_revision, GSM8K_REVISION, MATH500_REVISION]:
    if not re.fullmatch(r'[0-9a-f]{40}', revision):
        raise ValueError('Every repository revision must be a configured full SHA')

revisions = {
    'orca_dataset_id': ORCA_DATASET_ID,
    'orca_revision': orca_revision,
    'numina_dataset_id': NUMINA_DATASET_ID,
    'numina_revision': numina_revision,
    'tokenizer_id': TOKENIZER_ID,
    'tokenizer_revision': tokenizer_revision,
}
with open(REPORT_DIR / 'repository_revisions.json', 'w', encoding='utf-8') as f:
    json.dump(revisions, f, indent=2)

orca_raw = load_dataset(
    ORCA_DATASET_ID, split='train', revision=orca_revision,
    cache_dir=str(HF_CACHE_DIR), keep_in_memory=False
)
numina_raw = load_dataset(
    NUMINA_DATASET_ID, split='train', revision=numina_revision,
    cache_dir=str(HF_CACHE_DIR), keep_in_memory=False
)

full_counts = {'orca': len(orca_raw), 'numina': len(numina_raw)}
# Preserve original row positions before pilot shuffling so row IDs stay identical in pilot and full runs.
orca_raw = orca_raw.add_column('__original_index', np.arange(len(orca_raw), dtype=np.int64))
numina_raw = numina_raw.add_column('__original_index', np.arange(len(numina_raw), dtype=np.int64))

if RUN_MODE == 'pilot':
    orca_raw = orca_raw.shuffle(seed=SPLIT_SEED).select(range(min(PILOT_ROWS_PER_DATASET, len(orca_raw))))
    numina_raw = numina_raw.shuffle(seed=SPLIT_SEED).select(range(min(PILOT_ROWS_PER_DATASET, len(numina_raw))))

print('Repository revisions:', revisions)
print('Full published rows:', full_counts)
print('Rows used in this run:', {'orca': len(orca_raw), 'numina': len(numina_raw)})
print('Orca columns:', orca_raw.column_names)
print('Numina columns:', numina_raw.column_names)

6 D1 Dataset inventory and the 25-example inspection sheets

In [ ]:
inventory = [
    {
        'dataset_id': ORCA_DATASET_ID, 'revision': orca_revision, 'license_claim_in_v1': 'MIT',
        'published_train_rows': full_counts['orca'], 'columns': orca_raw.column_names,
        'problem_field': 'question', 'solution_field': 'answer', 'final_answer_field': None,
    },
    {
        'dataset_id': NUMINA_DATASET_ID, 'revision': numina_revision, 'license_claim_in_v1': 'Apache-2.0',
        'published_train_rows': full_counts['numina'], 'columns': numina_raw.column_names,
        'problem_field': 'problem', 'solution_field': 'solution', 'final_answer_field': 'answer',
    },
]
# Record license-card provenance at the exact dataset revision; never infer a license.
for item in inventory:
    info = api.dataset_info(item['dataset_id'], revision=item['revision'])
    card = info.card_data.to_dict() if info.card_data is not None else {}
    item['license_from_card'] = card.get('license', 'unspecified')
    item['license_provenance'] = {
        'repository_revision': item['revision'],
        'card_url': f"https://huggingface.co/datasets/{item['dataset_id']}/blob/{item['revision']}/README.md",
        'card_metadata': card,
    }
with open(REPORT_DIR / 'dataset_inventory.json', 'w', encoding='utf-8') as f:
    json.dump(inventory, f, indent=2)



7 D2 and D3 Canonical schema and quality flags

In [ ]:
def stable_row_id(dataset_id, revision, original_index):
    return sha256_text(f'{dataset_id}|{revision}|train|{original_index}')

def canonicalize_orca(row, idx):
    original_idx = int(row.get('__original_index', idx))
    problem_raw = raw_text(row.get('question', ''))
    solution_raw = raw_text(row.get('answer', ''))
    problem = clean_text(problem_raw)
    solution = clean_text(solution_raw)
    extraction = extract_answer_metadata(problem, solution)
    final_answer, answer_status = extraction.candidate, extraction.mode
    flags, review_flags = [], []
    if not problem: flags.append('empty_problem')
    if not solution: flags.append('missing_solution')
    if answer_status == 'missing': flags.append('missing_final_answer')
    if answer_status.startswith('ambiguous'): flags.append('ambiguous_final_answer')
    if answer_status == 'terminal_single_number_review': review_flags.append('heuristic_answer_extraction_review')
    if CJK_RE.search(problem) or CJK_RE.search(solution): flags.append('non_english_text')
    flags.extend(diagram_flags(problem, solution))
    flags.extend(target_quality_flags(problem, solution, final_answer, answer_status))
    flags = list(dict.fromkeys(flags))
    problem_hash = sha256_text(normalize_problem_for_matching(problem))
    row_id = stable_row_id(ORCA_DATASET_ID, orca_revision, original_idx)
    return {
        'row_id': row_id, 'dataset_id': 'orca', 'dataset_revision': orca_revision,
        'original_row_id': str(original_idx), 'original_source': 'microsoft_orca_math',
        'problem_raw': problem_raw, 'solution_raw': solution_raw,
        'problem': problem, 'solution': solution, 'final_answer': final_answer,
        'answer_parse_status': answer_status, 'correctness_status': 'unknown',
        'answer_extraction_json': json.dumps(extraction.to_dict(), ensure_ascii=False),
        'answer_evidence_start': extraction.evidence_start, 'answer_evidence_end': extraction.evidence_end,
        'answer_evidence_text': extraction.evidence_text, 'answer_confidence': extraction.confidence,
        'answer_type': answer_type(final_answer), 'answer_ambiguity_reason': extraction.ambiguity_reason,
        'answer_evidence_basis': extraction.evidence_basis,
        'extraction_consistency_status': extraction_consistency_status(extraction, final_answer),
        'solution_completeness_status': 'unknown',
        'source_fields_origin': row.get('__source_fields_origin', 'pinned_upstream'),
        'question_type': 'math-word-problem', 'problem_type': '', 'synthetic': True,
        'quality_flags': flags, 'review_flags': review_flags, 'eligible': len(flags) == 0,
        'answer_key': answer_key(final_answer),
        'problem_hash': problem_hash, 'problem_group_id': 'exact_' + problem_hash[:24],
        'split': 'candidate' if len(flags) == 0 else 'quarantine',
        'problem_char_length': len(problem), 'solution_char_length': len(solution),
    }

def canonicalize_numina(row, idx):
    original_idx = int(row.get('__original_index', idx))
    problem_raw = raw_text(row.get('problem', ''))
    solution_raw = raw_text(row.get('solution', ''))
    problem = clean_text(problem_raw)
    solution = clean_text(solution_raw)
    final_answer = compact_answer(row.get('answer', ''))
    extraction = extract_answer_metadata(problem, solution)
    question_type = clean_text(row.get('question_type', ''))
    original_source = clean_text(row.get('source', ''))
    synthetic = bool(row.get('synthetic', False))
    problem_valid = clean_text(row.get('problem_is_valid', '')).casefold()
    solution_valid = clean_text(row.get('solution_is_valid', '')).casefold()
    answer_status = 'parsed' if final_answer.casefold() not in MISSING_ANSWER_VALUES else 'missing'
    flags, review_flags = [], []
    if not problem: flags.append('empty_problem')
    if not solution: flags.append('missing_solution')
    if question_type.casefold() != 'math-word-problem': flags.append('outside_math_word_problem_metadata_scope')
    if answer_status != 'parsed': flags.append('missing_or_sentinel_final_answer')
    if SUSPECT_ANSWER_RE.search(final_answer): flags.append('suspect_answer_format')
    if problem_valid not in {'yes', 'true'}: flags.append('numina_problem_not_valid')
    if solution_valid not in {'yes', 'true'}: flags.append('numina_solution_not_valid')
    if original_source.casefold().replace('-', '_').replace(' ', '_') == 'orca_math':
        flags.append('embedded_orca_source_removed')
    if EXCLUDE_SYNTHETIC_NUMINA and synthetic:
        flags.append('synthetic_removed_by_team_policy')
    flags.extend(diagram_flags(problem, solution))
    if CJK_RE.search(problem) or CJK_RE.search(solution): flags.append('non_english_text')
    if answer_status == 'parsed' and squash_answer(final_answer) not in squash_answer(solution):
        review_flags.append('final_answer_not_literal_in_solution')
    flags.extend(target_quality_flags(problem, solution, final_answer, answer_status))
    flags = list(dict.fromkeys(flags))
    problem_hash = sha256_text(normalize_problem_for_matching(problem))
    row_id = stable_row_id(NUMINA_DATASET_ID, numina_revision, original_idx)
    return {
        'row_id': row_id, 'dataset_id': 'numina', 'dataset_revision': numina_revision,
        'original_row_id': str(original_idx), 'original_source': original_source,
        'problem_raw': problem_raw, 'solution_raw': solution_raw,
        'problem': problem, 'solution': solution, 'final_answer': final_answer,
        'answer_parse_status': answer_status, 'correctness_status': 'unknown',
        'answer_extraction_json': json.dumps(extraction.to_dict(), ensure_ascii=False),
        'answer_evidence_start': extraction.evidence_start, 'answer_evidence_end': extraction.evidence_end,
        'answer_evidence_text': extraction.evidence_text, 'answer_confidence': extraction.confidence,
        'answer_type': answer_type(final_answer), 'answer_ambiguity_reason': extraction.ambiguity_reason,
        'answer_evidence_basis': extraction.evidence_basis,
        'extraction_consistency_status': extraction_consistency_status(extraction, final_answer),
        'solution_completeness_status': 'unknown',
        'source_fields_origin': row.get('__source_fields_origin', 'pinned_upstream'),
        'provided_answer_raw': raw_text(row.get('answer', '')),
        'numina_problem_validity_raw': raw_text(row.get('problem_is_valid', '')),
        'numina_solution_validity_raw': raw_text(row.get('solution_is_valid', '')),
        'question_type': question_type, 'problem_type': clean_text(row.get('problem_type', '')),
        'synthetic': synthetic, 'quality_flags': flags, 'review_flags': review_flags, 'eligible': len(flags) == 0,
        'answer_key': answer_key(final_answer),
        'problem_hash': problem_hash, 'problem_group_id': 'exact_' + problem_hash[:24],
        'split': 'candidate' if len(flags) == 0 else 'quarantine',
        'problem_char_length': len(problem), 'solution_char_length': len(solution),
    }

def make_inspection_sheet(dataset, dataset_name, problem_col, solution_col, answer_col=None, n=25):
    rng = random.Random(SPLIT_SEED)
    indexes = rng.sample(range(len(dataset)), k=min(n, len(dataset)))
    rows = []
    for idx in indexes:
        item = dataset[idx]
        rows.append({
            'dataset': dataset_name,
            'original_index': int(item.get('__original_index', idx)),
            'problem': item.get(problem_col, ''),
            'solution': item.get(solution_col, ''),
            'provided_final_answer': item.get(answer_col, '') if answer_col else '',
            'manual_solution_complete_yes_no': '',
            'manual_final_answer_clear_yes_no': '',
            'manual_needs_missing_diagram_yes_no': '',
            'manual_keep_quarantine': '',
            'manual_notes': '',
        })
    out = pd.DataFrame(rows)
    out = save_review_sheet(out, REPORT_DIR / f'{dataset_name}_inspection_25.csv')
    return out

orca_inspection = make_inspection_sheet(orca_raw, 'orca', 'question', 'answer')
numina_inspection = make_inspection_sheet(numina_raw, 'numina', 'problem', 'solution', 'answer')
print('Created:', REPORT_DIR / 'orca_inspection_25.csv')
print('Created:', REPORT_DIR / 'numina_inspection_25.csv')
display(orca_inspection[['original_index', 'problem', 'solution']].head(3))
display(numina_inspection[['original_index', 'problem', 'solution', 'provided_final_answer']].head(3))

In [ ]:
NUM_PROC = None if os.name == 'nt' else min(2, os.cpu_count() or 1)
orca_canonical = orca_raw.map(
    canonicalize_orca, with_indices=True, remove_columns=orca_raw.column_names,
    num_proc=NUM_PROC, desc='Canonicalizing Orca'
)
numina_canonical = numina_raw.map(
    canonicalize_numina, with_indices=True, remove_columns=numina_raw.column_names,
    num_proc=NUM_PROC, desc='Canonicalizing Numina'
)

def flag_counter(dataset, column):
    counts = Counter()
    for flags in tqdm(dataset[column], desc=f'Counting {column}'):
        if not flags:
            counts['retained_without_flags'] += 1
        for flag in flags:
            counts[flag] += 1
    return counts

quality_rows, review_rows = [], []
for name, dataset in [('orca', orca_canonical), ('numina', numina_canonical)]:
    counts = flag_counter(dataset, 'quality_flags')
    quality_rows.extend({'dataset': name, 'flag': key, 'count': value} for key, value in counts.items())
    review_counts = flag_counter(dataset, 'review_flags')
    review_rows.extend({'dataset': name, 'review_flag': key, 'count': value} for key, value in review_counts.items())
pd.DataFrame(quality_rows).sort_values(['dataset', 'count'], ascending=[True, False]).to_csv(
    REPORT_DIR / 'quality_audit.csv', index=False
)
pd.DataFrame(review_rows).sort_values(['dataset', 'count'], ascending=[True, False]).to_csv(
    REPORT_DIR / 'review_flags_audit.csv', index=False
)

# Focus the human check on risky last-sentence guesses, then add high-confidence examples.
orca_heuristic_review = orca_canonical.filter(
    lambda row: 'heuristic_answer_extraction_review' in row['review_flags'],
    desc='Selecting heuristic Orca extractions'
).shuffle(seed=SPLIT_SEED)
heuristic_take = min(20, len(orca_heuristic_review))
orca_high_confidence_review = orca_canonical.filter(
    lambda row: row['answer_parse_status'] in {
        'explicit_final_marker', 'gsm_marker', 'boxed', 'terminal_answer_clause'
    },
    desc='Selecting marked Orca extractions'
).shuffle(seed=SPLIT_SEED + 1)
high_confidence_take = min(25 - heuristic_take, len(orca_high_confidence_review))
review_parts = []
if heuristic_take:
    review_parts.append(orca_heuristic_review.select(range(heuristic_take)))
if high_confidence_take:
    review_parts.append(orca_high_confidence_review.select(range(high_confidence_take)))
orca_review_source = concatenate_datasets(review_parts) if review_parts else orca_canonical.select(range(0))
orca_extraction_review = orca_review_source.select_columns(
    ['row_id', 'original_row_id', 'problem', 'solution', 'final_answer', 'answer_parse_status']
).to_pandas()
orca_extraction_review['solution_tail'] = orca_extraction_review['solution'].str[-500:]
# Full source solution is retained for completeness and mathematical review.
orca_extraction_review['manual_extraction_correct_yes_no'] = ''
orca_extraction_review['manual_corrected_answer'] = ''
orca_extraction_review['manual_notes'] = ''
orca_extraction_review = save_review_sheet(orca_extraction_review, REPORT_DIR / 'orca_answer_extraction_review_25.csv')

print('Eligible Orca:', sum(orca_canonical['eligible']), '/', len(orca_canonical))
print('Eligible Numina:', sum(numina_canonical['eligible']), '/', len(numina_canonical))
display(pd.read_csv(REPORT_DIR / 'quality_audit.csv').head(20))
display(orca_extraction_review.head(5))

In [ ]:
RUN_TAG = 'pilot' if RUN_MODE == 'pilot' else 'full'

def save_dataset_parquet(dataset, final_path):
    temp_path = LOCAL_WORK_DIR / final_path.name
    dataset.to_parquet(str(temp_path))
    shutil.copy2(temp_path, final_path)
    print('Saved:', final_path)

save_dataset_parquet(orca_canonical, PROCESSED_DIR / f'orca_canonical_{RUN_TAG}.parquet')
save_dataset_parquet(numina_canonical, PROCESSED_DIR / f'numina_canonical_{RUN_TAG}.parquet')

8 D4 Exact duplicates and a bounded near-duplicate candidate universe

In [ ]:
orca_eligible = orca_canonical.filter(lambda row: row['eligible'], num_proc=NUM_PROC, desc='Filtering eligible Orca')
numina_eligible = numina_canonical.filter(lambda row: row['eligible'], num_proc=NUM_PROC, desc='Filtering eligible Numina')
eligible_all = concatenate_datasets([orca_eligible, numina_eligible])

meta_cols = [
    'row_id', 'dataset_id', 'original_source', 'problem_hash', 'problem_group_id',
    'answer_key', 'final_answer', 'solution_char_length'
]
eligible_meta = eligible_all.select_columns(meta_cols).to_pandas()
duplicate_stats = eligible_meta.groupby('problem_hash').agg(
    duplicate_count=('row_id', 'size'),
    answer_count=('answer_key', 'nunique'),
    datasets=('dataset_id', lambda values: ','.join(sorted(set(values)))),
)
duplicate_stats = duplicate_stats[duplicate_stats['duplicate_count'] > 1].copy()
shared_checker = None
if SHARED_EQUIVALENCE_PATH.exists():
    import importlib.util
    checker_spec = importlib.util.spec_from_file_location('team_equivalence', SHARED_EQUIVALENCE_PATH)
    checker_module = importlib.util.module_from_spec(checker_spec)
    checker_spec.loader.exec_module(checker_module)
    shared_checker = checker_module.mathematically_equal
else:
    INTEGRATION_BLOCKERS.append('Shared evaluator equivalence checker missing for symbolic conflict audit')
equivalence_audit = []
unresolved_hashes = set()
for problem_hash in duplicate_stats.index[duplicate_stats['answer_count'] > 1]:
    answers = eligible_meta.loc[eligible_meta['problem_hash'].eq(problem_hash), 'final_answer'].unique()
    decisions = [answers_equivalent(answers[0], answer, shared_checker) for answer in answers[1:]]
    if all(value is True for value in decisions):
        duplicate_stats.loc[problem_hash, 'answer_count'] = 1
        state = 'equivalent'
    elif any(value is False for value in decisions):
        state = 'proven_scalar_or_checker_disagreement'
    else:
        state = 'unresolved_equivalence_quarantine'
        unresolved_hashes.add(problem_hash)
    equivalence_audit.append({'problem_hash': problem_hash, 'answers': json.dumps(answers.tolist()), 'equivalence_status': state})
pd.DataFrame(equivalence_audit, columns=['problem_hash','answers','equivalence_status']).to_csv(
    REPORT_DIR / 'exact_answer_equivalence_audit.csv', index=False)
duplicate_stats['conflicting_answers'] = duplicate_stats['answer_count'] > 1
duplicate_stats['equivalence_unresolved'] = duplicate_stats.index.isin(unresolved_hashes)
duplicate_hashes = set(duplicate_stats.index)
conflicting_hashes = set(duplicate_stats.index[duplicate_stats['conflicting_answers']])
exact_report = eligible_meta[eligible_meta['problem_hash'].isin(duplicate_hashes)].copy()
exact_report = exact_report.merge(duplicate_stats, on='problem_hash', how='left')
exact_report.sort_values(['problem_hash', 'row_id']).to_csv(REPORT_DIR / 'exact_duplicates.csv', index=False)
exact_report[exact_report['conflicting_answers']].to_csv(REPORT_DIR / 'conflicting_exact_answers.csv', index=False)

# Quarantine the whole exact-match group when its answers conflict.
nonconflicting_meta = eligible_meta[~eligible_meta['problem_hash'].isin(conflicting_hashes)].copy()
canonical_rows = nonconflicting_meta.sort_values('row_id').drop_duplicates('problem_hash', keep='first')
canonical_ids = set(canonical_rows['row_id'])
conflicting_row_ids = set(eligible_meta.loc[eligible_meta['problem_hash'].isin(conflicting_hashes), 'row_id'])
exact_alternative_ids = set(nonconflicting_meta.loc[~nonconflicting_meta['row_id'].isin(canonical_ids), 'row_id'])
exact_unique = eligible_all.filter(
    lambda row: row['row_id'] in canonical_ids and row['problem_hash'] not in conflicting_hashes,
    desc='Keeping one representative per exact problem group'
)

def deterministic_source_sample(dataset, n, seed):
    n = min(n, len(dataset))
    return dataset.shuffle(seed=seed).select(range(n))

orca_unique = exact_unique.filter(lambda row: row['dataset_id'] == 'orca', desc='Selecting unique Orca')
numina_unique = exact_unique.filter(lambda row: row['dataset_id'] == 'numina', desc='Selecting unique Numina')
universe_per_source = min(
    CANDIDATE_UNIVERSE_PER_SOURCE if RUN_MODE == 'full' else 5_000,
    len(orca_unique), len(numina_unique)
)
candidate_universe = concatenate_datasets([
    deterministic_source_sample(orca_unique, universe_per_source, SELECTION_SEED),
    deterministic_source_sample(numina_unique, universe_per_source, SELECTION_SEED),
])
candidate_df = candidate_universe.to_pandas()
candidate_universe_ids = set(candidate_df['row_id'])

print('Exact duplicate rows reported:', len(exact_report))
print('Conflicting-answer exact groups quarantined:', len(conflicting_hashes))
print('Exact-unique eligible rows:', len(exact_unique))
print('Near-duplicate candidate universe:', candidate_df['dataset_id'].value_counts().to_dict())

9 D5 Remove overlap with Person 3's reserved evaluation problems

In [ ]:
# Roles/provenance come from metadata, not filenames. Person 3 owns these data.
protection_bundle = load_protection_inputs(
    EVAL_MANIFEST_PATH, INPUT_DIR / 'reserved_evaluation_metadata.json',
    DEV_MANIFEST_PATH, DEV_METADATA_PATH,
)
reserved_test_df = protection_bundle['reserved']
development_df = protection_bundle['development']
eval_df = protection_bundle['questions'].copy()
protection_inputs = protection_bundle['inputs']
DEVELOPMENT_PROTECTED = protection_bundle['development_protected']
if not DEVELOPMENT_PROTECTED:
    INTEGRATION_BLOCKERS.append('Person 3 frozen development coverage pending')
    print('Person 1 data preparation continues; development protection remains pending.')
protection_scope = export_protection_inputs(protection_bundle, PROCESSED_DIR, REPORT_DIR)
eval_df['normalized_problem'] = eval_df['problem'].map(normalize_problem_for_matching)
eval_hashes = set(eval_df['problem_hash'])

checked_candidate_count = len(candidate_df)
candidate_df['exact_eval_overlap'] = candidate_df['problem_hash'].isin(eval_hashes)
exact_overlap_report = candidate_df[candidate_df['exact_eval_overlap']].copy()
exact_eval_overlap_ids = set(exact_overlap_report['row_id'])
exact_overlap_report.to_csv(REPORT_DIR / 'evaluation_exact_overlap.csv', index=False)
candidate_df = candidate_df[~candidate_df['exact_eval_overlap']].reset_index(drop=True)
protection_scope['exact_candidate_checks'] = {
    'checked_rows': checked_candidate_count, 'overlapping_rows_removed': len(exact_overlap_report),
}
protection_scope['overlap_checks_status'] = 'exact_candidates_checked; near_candidates_pending'
(REPORT_DIR / 'evaluation_protection_scope.json').write_text(json.dumps(protection_scope, indent=2), encoding='utf-8')
print('Exact protected development/test overlaps removed:', len(exact_overlap_report))


10 Generate and inspect near-duplicate candidates

In [ ]:
def word_shingles(text, size=5):
    tokens = normalize_problem_for_matching(text).split()
    if not tokens:
        return set()
    if len(tokens) < size:
        return {' '.join(tokens)}
    return {' '.join(tokens[i:i + size]) for i in range(len(tokens) - size + 1)}

def make_minhash(shingles, num_perm=MINHASH_PERMUTATIONS):
    mh = MinHash(num_perm=num_perm)
    for shingle in sorted(shingles):
        mh.update(shingle.encode('utf-8'))
    return mh

lsh = MinHashLSH(threshold=LSH_CANDIDATE_THRESHOLD, num_perm=MINHASH_PERMUTATIONS)
shingle_cache = {}
near_pairs = {}

for row in tqdm(candidate_df.itertuples(index=False), total=len(candidate_df), desc='Finding near duplicates'):
    shingles = word_shingles(row.problem)
    if not shingles:
        continue
    mh = make_minhash(shingles)
    for other_id in lsh.query(mh):
        other_shingles = shingle_cache[other_id]
        union = shingles | other_shingles
        similarity = len(shingles & other_shingles) / len(union) if union else 1.0
        key = tuple(sorted((row.row_id, other_id)))
        near_pairs[key] = max(similarity, near_pairs.get(key, 0.0))
    lsh.insert(row.row_id, mh)
    shingle_cache[row.row_id] = shingles

near_pairs_df = pd.DataFrame(
    [{'row_id_a': a, 'row_id_b': b, 'jaccard_5gram': score} for (a, b), score in near_pairs.items()]
)
if len(near_pairs_df):
    lookup = candidate_df.set_index('row_id')[['dataset_id', 'problem']]
    near_pairs_df['dataset_a'] = near_pairs_df['row_id_a'].map(lookup['dataset_id'])
    near_pairs_df['dataset_b'] = near_pairs_df['row_id_b'].map(lookup['dataset_id'])
    near_pairs_df['problem_a'] = near_pairs_df['row_id_a'].map(lookup['problem'])
    near_pairs_df['problem_b'] = near_pairs_df['row_id_b'].map(lookup['problem'])
    near_pairs_df = near_pairs_df.sort_values('jaccard_5gram', ascending=False)
else:
    near_pairs_df = pd.DataFrame(columns=[
        'row_id_a', 'row_id_b', 'jaccard_5gram', 'dataset_a', 'dataset_b', 'problem_a', 'problem_b'
    ])

near_pairs_df.to_csv(REPORT_DIR / 'near_duplicate_candidates.csv', index=False)
high = near_pairs_df.head(50)
borderline = near_pairs_df.iloc[(near_pairs_df['jaccard_5gram'] - NEAR_DUPLICATE_THRESHOLD).abs().argsort()[:50]] if len(near_pairs_df) else near_pairs_df
review = pd.concat([high, borderline]).drop_duplicates(['row_id_a', 'row_id_b']).copy()
review['manual_same_problem_yes_no'] = ''
review['manual_notes'] = ''
review['manual_overlap_kind'] = ''
review['manual_policy_action'] = ''
review = save_review_sheet(review, REPORT_DIR / 'near_duplicate_review_sample.csv')

# Generate train-versus-evaluation candidates separately for threshold review.
eval_candidate_rows = []
if len(eval_df):
    candidate_problem_lookup = candidate_df.set_index('row_id')['problem']
    for erow in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc='Finding evaluation near-overlap candidates'):
        eval_shingles = word_shingles(erow.problem)
        if not eval_shingles:
            continue
        eval_mh = make_minhash(eval_shingles)
        for candidate_id in lsh.query(eval_mh):
            train_shingles = shingle_cache[candidate_id]
            union = eval_shingles | train_shingles
            similarity = len(eval_shingles & train_shingles) / len(union) if union else 1.0
            eval_candidate_rows.append({
                'eval_set_id': erow.set_id, 'eval_problem_id': erow.problem_id,
                'training_row_id': candidate_id, 'jaccard_5gram': similarity,
                'eval_problem': erow.problem, 'training_problem': candidate_problem_lookup[candidate_id],
            })
eval_pair_candidates_df = pd.DataFrame(eval_candidate_rows)
if len(eval_pair_candidates_df):
    eval_pair_candidates_df = eval_pair_candidates_df.sort_values('jaccard_5gram', ascending=False)
else:
    eval_pair_candidates_df = pd.DataFrame(columns=[
        'eval_set_id', 'eval_problem_id', 'training_row_id', 'jaccard_5gram',
        'eval_problem', 'training_problem'
    ])
eval_pair_candidates_df.to_csv(REPORT_DIR / 'evaluation_near_candidates.csv', index=False)
eval_high = eval_pair_candidates_df.head(50)
eval_borderline = eval_pair_candidates_df.iloc[(eval_pair_candidates_df['jaccard_5gram'] - DECONTAMINATION_THRESHOLD).abs().argsort()[:50]] if len(eval_pair_candidates_df) else eval_pair_candidates_df
eval_review = pd.concat([eval_high, eval_borderline]).drop_duplicates(
    ['eval_set_id', 'eval_problem_id', 'training_row_id']
).copy()
eval_review['manual_same_problem_yes_no'] = ''
eval_review['manual_notes'] = ''
eval_review['manual_overlap_kind'] = ''
eval_review['manual_policy_action'] = ''
eval_review = save_review_sheet(eval_review, REPORT_DIR / 'evaluation_near_review_sample.csv')
print('Near-duplicate candidate pairs:', len(near_pairs_df))
print('Evaluation near-overlap candidate pairs:', len(eval_pair_candidates_df))
print('Review both threshold files in:', REPORT_DIR)
def template_signature(problem):
    text = normalize_problem_for_matching(problem)
    text = NUMBER_RE.sub('<number>', text)
    return re.sub(r'(letters of|word)\s+[a-z]+', r'\1 <word>', text)

missed_rows = []
known_pairs = set(near_pairs)
template_groups = defaultdict(list)
for row in candidate_df.itertuples(index=False):
    key = template_signature(row.problem)
    group = template_groups[key]
    if group and len(missed_rows) < 100:
        other = group[0]
        pair_key = tuple(sorted([row.row_id, other.row_id]))
        if pair_key not in known_pairs:
            a, b = word_shingles(row.problem), word_shingles(other.problem)
            missed_rows.append({'row_id_a': other.row_id, 'row_id_b': row.row_id,
                'problem_a': other.problem, 'problem_b': row.problem,
                'jaccard_5gram': len(a & b) / len(a | b) if a | b else 1.0,
                'search_method': 'parameter_normalized_signature'})
    group.append(row)
template_train = {key: group[0] for key, group in template_groups.items()}
known_eval = {(row.eval_set_id, row.eval_problem_id, row.training_row_id) for row in eval_pair_candidates_df.itertuples(index=False)}
for row in eval_df.itertuples(index=False):
    other = template_train.get(template_signature(row.problem))
    if other is not None and (row.set_id, row.problem_id, other.row_id) not in known_eval:
        a, b = word_shingles(row.problem), word_shingles(other.problem)
        missed_rows.append({'row_id_a': f'{row.set_id}|{row.problem_id}', 'row_id_b': other.row_id,
            'problem_a': row.problem, 'problem_b': other.problem,
            'jaccard_5gram': len(a & b) / len(a | b) if a | b else 1.0,
            'search_method': 'protected_parameter_normalized_signature'})
missed_review = pd.DataFrame(missed_rows, columns=['row_id_a','row_id_b','problem_a','problem_b','jaccard_5gram','search_method'])
missed_review['manual_same_problem_yes_no'] = ''
missed_review['manual_overlap_kind'] = ''
missed_review['manual_policy_action'] = ''
missed_review['manual_notes'] = ''
save_review_sheet(missed_review, REPORT_DIR / 'missed_match_review.csv')


11 Apply the reviewed near-duplicate threshold and near-evaluation removal

In [ ]:
# Threshold removals are provisional until saved reviews approve this exact policy.
# Conservative lexical variants may have different answers; counts are not claims of duplicate instances.
if REVIEW_POLICY['near_duplicate_threshold'] != NEAR_DUPLICATE_THRESHOLD or REVIEW_POLICY['decontamination_threshold'] != DECONTAMINATION_THRESHOLD:
    raise ValueError('Threshold configuration disagrees with the versioned policy')
log_decision(
    f'near_duplicate_threshold={NEAR_DUPLICATE_THRESHOLD}; '
    f'decontamination_threshold={DECONTAMINATION_THRESHOLD}; minhash_permutations={MINHASH_PERMUTATIONS}'
)

class UnionFind:
    def __init__(self, items):
        self.parent = {item: item for item in items}
    def find(self, item):
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        root, child = sorted([ra, rb])
        self.parent[child] = root

uf = UnionFind(candidate_df['row_id'])
applied_pairs = near_pairs_df[near_pairs_df['jaccard_5gram'] >= NEAR_DUPLICATE_THRESHOLD]
for pair in applied_pairs.itertuples(index=False):
    uf.union(pair.row_id_a, pair.row_id_b)

candidate_df['near_group_root'] = candidate_df['row_id'].map(uf.find)
candidate_df['problem_group_id'] = candidate_df['near_group_root'].map(
    lambda value: 'near_' + sha256_text(value)[:24]
)

# Apply the separately reviewed evaluation decontamination threshold.
eval_near_report = eval_pair_candidates_df[
    eval_pair_candidates_df['jaccard_5gram'] >= DECONTAMINATION_THRESHOLD
].copy()
eval_near_report.to_csv(REPORT_DIR / 'evaluation_near_overlap.csv', index=False)
near_eval_ids = set(eval_near_report['training_row_id']) if len(eval_near_report) else set()
near_eval_groups = set(candidate_df.loc[candidate_df['row_id'].isin(near_eval_ids), 'problem_group_id'])
near_eval_removed_ids = set(candidate_df.loc[candidate_df['problem_group_id'].isin(near_eval_groups), 'row_id'])
candidate_df = candidate_df[~candidate_df['problem_group_id'].isin(near_eval_groups)].copy()

# Keep one deterministic representative from every near-duplicate group.
candidate_df = candidate_df.sort_values('row_id')
near_representative_ids = set(candidate_df.drop_duplicates('problem_group_id', keep='first')['row_id'])
near_duplicate_alternative_ids = set(candidate_df.loc[~candidate_df['row_id'].isin(near_representative_ids), 'row_id'])
candidate_df = candidate_df[candidate_df['row_id'].isin(near_representative_ids)].reset_index(drop=True)
print('Near-duplicate pairs applied:', len(applied_pairs))
print('Near-evaluation groups removed:', len(near_eval_groups))
print('Clean representatives remaining:', len(candidate_df))
# Record the precise protection-check scope; legacy matching remains diagnostic.
protection_scope['near_candidate_checks'] = {
    'matching_policy': REVIEW_POLICY['overlap_scope'],
    'checked_candidate_scope_per_source': universe_per_source,
    'protected_question_entries': len(eval_df),
    'candidate_pairs': len(eval_pair_candidates_df),
    'removed_groups': len(near_eval_groups),
    'recall_guaranteed': False,
}
protection_scope['overlap_checks_status'] = 'bounded_exact_and_approximate_near_candidate_checks_completed'
(REPORT_DIR / 'evaluation_protection_scope.json').write_text(json.dumps(protection_scope, indent=2), encoding='utf-8')


12 D6 Format and tokenize the exact training text

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID, revision=tokenizer_revision, use_fast=True, cache_dir=str(HF_CACHE_DIR)
)
verify_tokenizer(tokenizer)
if TOKENIZER_ID != 'Qwen/Qwen2.5-0.5B' or tokenizer_revision != PINNED_TOKENIZER_REVISION:
    raise ValueError('Tokenizer identity/revision differs from the reference contract')
run_config_dir = RUN_ROOT / 'configs'
run_config_dir.mkdir(exist_ok=True)
asset_directory = HF_CACHE_DIR / 'models--Qwen--Qwen2.5-0.5B/snapshots' / tokenizer_revision
contract_spec = formatting_contract(tokenizer, asset_directory=asset_directory, max_total_tokens=MAX_TOTAL_TOKENS)
contract_spec['reference_encoder_sha256'] = sha256_text(CONTRACT_SOURCE)
(run_config_dir / 'formatting_contract.json').write_text(json.dumps(contract_spec, indent=2), encoding='utf-8')

TEMPLATE_HASH = sha256_text(PROMPT_TEMPLATE)
prompt_spec = {
    'schema_version': PROMPT_SPEC_VERSION, 'template': PROMPT_TEMPLATE,
    'template_sha256': TEMPLATE_HASH, 'control_version': CONTROL_VERSION,
    'tokenizer_id': TOKENIZER_ID, 'tokenizer_revision': tokenizer_revision,
    'add_special_tokens': False, 'append_eos': True, 'eos_token_id': tokenizer.eos_token_id,
    'supervise_prompt': False, 'supervise_completion_and_eos': True,
    'max_total_tokens': MAX_TOTAL_TOKENS,
    'manifest_field_mapping': {'id': 'row_id', 'group_id': 'problem_group_id',
                               'prompt': 'prompt_text', 'target': 'completion'},
}
(REPORT_DIR / 'prompt_specification.json').write_text(json.dumps(prompt_spec, indent=2), encoding='utf-8')
PROMPT_SPEC_HASH = file_sha256(REPORT_DIR / 'prompt_specification.json')
def batch_training_token_stats(frame, batch_size=TOKENIZATION_BATCH_SIZE):
    token_rows = []
    for start in tqdm(range(0, len(frame), batch_size), desc='Encoding candidate labels'):
        for row in frame.iloc[start:start + batch_size].to_dict('records'):
            try:
                # No truncation here: the whole encoded row is then excluded if overlength.
                encoded = encode_example(tokenizer, row['problem'], row['solution'], row['final_answer'], max_total_tokens=None)
                stats = {key: value for key, value in encoded.items() if key not in {'input_ids', 'attention_mask', 'labels'}}
                stats.update(target_encoding_error=False, target_encoding_error_reason='')
            except ValueError as error:
                stats = dict(prompt_text=render_prompt(row['problem']), completion='', prompt_tokens=0,
                    completion_tokens=0, eos_tokens=0, supervised_tokens=0, total_tokens=0,
                    completion_start=0, boundary_mismatch=False, boundary_crossing_token=isinstance(error, BoundaryError),
                    target_encoding_error=True, target_encoding_error_reason=str(error))
            token_rows.append(stats)
    return token_rows

token_rows = batch_training_token_stats(candidate_df)
token_stats_df = pd.DataFrame(token_rows)
candidate_df = pd.concat([candidate_df.reset_index(drop=True), token_stats_df], axis=1)
candidate_df['overlength'] = candidate_df['total_tokens'] > MAX_TOTAL_TOKENS
overlength_ids = set(candidate_df.loc[candidate_df['overlength'] | candidate_df['target_encoding_error'], 'row_id'])
candidate_df[candidate_df['target_encoding_error']].to_csv(REPORT_DIR / 'target_encoding_quarantine.csv', index=False)

# Failed boundary/consistency examples are quarantined as whole rows.

length_summary = (
    candidate_df[~candidate_df['target_encoding_error']].groupby('dataset_id')['total_tokens']
    .agg(
        count='count', mean='mean', p50=lambda x: x.quantile(.50),
        p90=lambda x: x.quantile(.90), p95=lambda x: x.quantile(.95),
        p99=lambda x: x.quantile(.99), maximum='max'
    ).reset_index()
)
overlength_summary = candidate_df.groupby('dataset_id')['overlength'].agg(['sum', 'mean']).reset_index()
overlength_summary = overlength_summary.rename(columns={'sum': 'overlength_count', 'mean': 'overlength_fraction'})
length_summary = length_summary.merge(overlength_summary, on='dataset_id')
length_summary.to_csv(REPORT_DIR / 'token_length_statistics.csv', index=False)
candidate_df[candidate_df['overlength']].to_csv(REPORT_DIR / 'overlength_quarantine.csv', index=False)

# Manual inspection required by the implementation plan: 10 formatted rows from each source.
formatted_review = pd.concat([
    candidate_df[(candidate_df['dataset_id'] == source) & (~candidate_df['target_encoding_error'])].head(10)
    for source in ['orca', 'numina']
])[['row_id', 'dataset_id', 'prompt_text', 'completion', 'prompt_tokens', 'completion_tokens', 'total_tokens']].copy()
formatted_review['manual_prompt_correct_yes_no'] = ''
formatted_review['manual_reasoning_preserved_yes_no'] = ''
formatted_review['manual_one_trailing_final_answer_yes_no'] = ''
formatted_review['manual_notes'] = ''
formatted_review['manual_solution_complete_yes_no'] = ''
formatted_review['manual_answer_solution_consistent_yes_no'] = ''
formatted_review['manual_mathematically_correct_yes_no'] = ''
formatted_review = save_review_sheet(formatted_review, REPORT_DIR / 'formatted_examples_review_20.csv')

display(length_summary)
if candidate_df['overlength'].mean() > 0.05:
    print('ACTION REQUIRED: more than 5% exceed 2,048 tokens. Discuss testing 4,096 with Person 2.')
print('Review 20 formatted examples in:', REPORT_DIR / 'formatted_examples_review_20.csv')

13 D7 Build the balanced working pool and nested manifests

In [ ]:
usable_df = candidate_df[(~candidate_df['overlength']) & (~candidate_df['boundary_crossing_token']) & (~candidate_df['target_encoding_error'])].copy()

def add_length_bins(frame):
    pieces = []
    for dataset_id, group in frame.groupby('dataset_id', sort=True):
        group = group.copy()
        group['length_bin'] = pd.qcut(
            group['supervised_tokens'].rank(method='first'), q=10, labels=False, duplicates='drop'
        ).astype(int)
        pieces.append(group)
    return pd.concat(pieces, ignore_index=True)

def stable_selection_key(row_id, seed):
    return sha256_text(f'{seed}|{row_id}')

def balanced_round_robin_order(frame, seed):
    frame = frame.copy()
    frame['selection_key'] = frame['row_id'].map(lambda x: stable_selection_key(x, seed))
    groups = {}
    for key, group in frame.groupby(['dataset_id', 'length_bin'], sort=True):
        groups[key] = list(group.sort_values('selection_key').index)
    positions = {key: 0 for key in groups}
    order = []
    while True:
        added = False
        for key in sorted(groups):
            pos = positions[key]
            if pos < len(groups[key]):
                order.append(groups[key][pos])
                positions[key] += 1
                added = True
        if not added:
            break
    return frame.loc[order].reset_index(drop=True)

usable_df = add_length_bins(usable_df)
ordered_df = balanced_round_robin_order(usable_df, SELECTION_SEED)

required_per_source = TARGET_PER_SOURCE if RUN_MODE == 'full' else min(2_000, TARGET_PER_SOURCE)
source_counts = ordered_df['dataset_id'].value_counts()
if any(source_counts.get(source, 0) < required_per_source for source in ['orca', 'numina']):
    raise ValueError(f'Insufficient usable examples for a balanced pool: {source_counts.to_dict()}')

# Rebalance to equal source totals, then restore source-by-bin round-robin ordering.
balanced_parts = [
    ordered_df[ordered_df['dataset_id'] == source].head(required_per_source)
    for source in sorted(ordered_df['dataset_id'].unique())
]
working_candidates = pd.concat(balanced_parts, ignore_index=True)
working_order = balanced_round_robin_order(working_candidates, SELECTION_SEED)
working_pool_size = min(TARGET_WORKING_POOL if RUN_MODE == 'full' else 4_000, len(working_order))
working_pool = working_order.head(working_pool_size).copy()
working_pool_ids = set(working_pool['row_id'])

manifest_columns = [
    'row_id', 'problem_group_id', 'dataset_id', 'original_source', 'length_bin',
    'problem', 'completion', 'final_answer', 'prompt_tokens', 'completion_tokens', 'supervised_tokens',
    'total_tokens', 'dataset_revision', 'original_row_id', 'solution', 'prompt_text',
    'prompt_spec_version', 'prompt_template_sha256', 'control_version', 'correctness_status',
    'answer_parse_status', 'artifact_status',
    'answer_extraction_json', 'answer_evidence_text', 'answer_evidence_start', 'answer_evidence_end',
    'answer_confidence', 'answer_type', 'answer_ambiguity_reason', 'answer_evidence_basis',
    'problem_raw', 'solution_raw', 'quality_flags', 'review_flags', 'eos_tokens',
    'extraction_consistency_status', 'solution_completeness_status', 'source_fields_origin'
]
for frame in [usable_df, working_pool]:
    frame['prompt_spec_version'] = PROMPT_SPEC_VERSION
    frame['prompt_template_sha256'] = TEMPLATE_HASH
    frame['control_version'] = CONTROL_VERSION
    frame['artifact_status'] = 'draft_pending_review_and_integration'
working_pool[manifest_columns].to_parquet(PROCESSED_DIR / f'train_pool_{RUN_TAG}.parquet', index=False)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def ordered_ids_sha256(row_ids):
    payload = '\n'.join(row_ids.astype(str).tolist()) + '\n'
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

def save_manifest(name, frame):
    out = frame[manifest_columns].copy()
    path = MANIFEST_DIR / f'{name}_{RUN_TAG}.jsonl'
    ids_path = MANIFEST_DIR / f'{name}_{RUN_TAG}_ids.txt'
    out.to_json(path, orient='records', lines=True, force_ascii=False)
    ids_path.write_text('\n'.join(out['row_id'].astype(str)) + '\n', encoding='utf-8')
    summary = {
        'name': name, 'run_tag': RUN_TAG, 'rows': len(out),
        'artifact_status': 'draft_pending_review_and_integration', 'run_fingerprint': RUN_FINGERPRINT,
        'development_protected': DEVELOPMENT_PROTECTED, 'token_budget_finalized': False,
        'prompt_spec_version': PROMPT_SPEC_VERSION, 'control_version': CONTROL_VERSION,
        'source_counts': out['dataset_id'].value_counts().to_dict(),
        'source_supervised_tokens': out.groupby('dataset_id')['supervised_tokens'].sum().to_dict(),
        'total_prompt_tokens': int(out['prompt_tokens'].sum()),
        'total_completion_tokens': int(out['completion_tokens'].sum()),
        'total_supervised_tokens': int(out['supervised_tokens'].sum()),
        'total_sequence_tokens': int(out['total_tokens'].sum()),
        'length_bin_counts': {
            f'{source}|{int(bin_id)}': int(count)
            for (source, bin_id), count in out.groupby(['dataset_id', 'length_bin']).size().items()
        },
        'sha256': file_sha256(path), 'ordered_ids_sha256': ordered_ids_sha256(out['row_id']),
        'ids_file_sha256': file_sha256(ids_path), 'selection_seed': SELECTION_SEED,
        'tokenizer_revision': tokenizer_revision, 'template_hash': TEMPLATE_HASH,
    }
    with open(MANIFEST_DIR / f'{name}_{RUN_TAG}_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)
    return out, summary

debug_32, debug_summary = save_manifest('debug_32', working_pool.head(32))
manifest_sizes = MANIFEST_SIZES
manifests = {}
summaries = {}
for size in manifest_sizes:
    if size <= len(working_pool):
        manifests[size], summaries[size] = save_manifest(f'train_n{size}', working_pool.head(size))

for smaller, larger in zip(manifest_sizes, manifest_sizes[1:]):
    if smaller in manifests and larger in manifests:
        assert set(manifests[smaller]['row_id']).issubset(set(manifests[larger]['row_id']))

assert working_pool['problem_group_id'].is_unique
print('Working pool:', len(working_pool))
print('Manifest summaries:')
display(pd.DataFrame(summaries).T[['rows', 'total_supervised_tokens', 'sha256']])

14 Validate the data handoff

In [ ]:
assert working_pool['row_id'].is_unique
assert working_pool['problem_group_id'].is_unique
assert working_pool['problem'].str.len().gt(0).all()
assert working_pool['completion'].str.len().gt(0).all()
assert working_pool['final_answer'].str.len().gt(0).all()
assert working_pool['total_tokens'].le(MAX_TOTAL_TOKENS).all()
assert (~working_pool['boundary_crossing_token']).all()
assert (~working_pool['target_encoding_error']).all()
assert not set(working_pool['problem_hash']) & eval_hashes
assert not set(working_pool['problem_group_id']) & near_eval_groups
if RUN_MODE == 'full':
    assert EVAL_MANIFEST_PATH.exists(), 'Full run requires Person 3 reserved_evaluation.csv.'

# Preserve a row-level disposition for every source record without converting the full text corpus to pandas.
audit_columns = [
    'row_id', 'dataset_id', 'original_row_id', 'original_source', 'problem_hash',
    'answer_parse_status', 'quality_flags', 'review_flags', 'eligible'
]
audit_base = concatenate_datasets([
    orca_canonical.select_columns(audit_columns),
    numina_canonical.select_columns(audit_columns),
])

def assign_final_status(row):
    row_id = row['row_id']
    if not row['eligible']:
        status = 'excluded_quality'
    elif row_id in conflicting_row_ids:
        status = 'conflicting_exact_answers'
    elif row_id in exact_alternative_ids:
        status = 'exact_duplicate_alternative'
    elif row_id in working_pool_ids:
        status = 'train_pool'
    elif row_id in exact_eval_overlap_ids:
        status = 'exact_evaluation_overlap'
    elif row_id in near_eval_removed_ids:
        status = 'near_evaluation_overlap_group'
    elif row_id in near_duplicate_alternative_ids:
        status = 'near_duplicate_alternative'
    elif row_id in overlength_ids:
        status = 'overlength_or_target_encoding_error'
    elif row_id in candidate_universe_ids:
        status = 'usable_candidate_not_selected'
    else:
        status = 'eligible_not_sampled'
    return {
        'final_status': status,
        'quality_flags_text': ';'.join(row['quality_flags'] or []),
        'review_flags_text': ';'.join(row['review_flags'] or []),
    }

row_audit = audit_base.map(assign_final_status, num_proc=None, desc='Assigning final row status')
row_audit_export = row_audit.remove_columns(['quality_flags', 'review_flags'])
save_dataset_parquet(row_audit_export, PROCESSED_DIR / f'row_status_audit_{RUN_TAG}.parquet')
final_status_counts = {key: int(value) for key, value in Counter(row_audit['final_status']).items()}

audit_summary = {
    'run_mode': RUN_MODE, 'artifact_status': 'draft_pending_review_and_integration',
    'development_protected': DEVELOPMENT_PROTECTED, 'review_policy': REVIEW_POLICY,
    'quality_count_semantics': 'nonexclusive_flags; not mutually exclusive row outcomes',
    'full_published_counts': full_counts,
    'rows_used_this_run': {'orca': len(orca_raw), 'numina': len(numina_raw)},
    'eligible_before_exact_dedup': len(eligible_all),
    'exact_duplicate_rows_reported': len(exact_report),
    'near_duplicate_scope_per_source': universe_per_source,
    'near_pairs_applied': len(applied_pairs),
    'exact_evaluation_overlaps_removed': len(exact_overlap_report),
    'near_evaluation_groups_removed': len(near_eval_groups),
    'conflicting_exact_answer_rows_removed': len(conflicting_row_ids),
    'overlength_or_boundary_rows_removed': len(overlength_ids),
    'evaluation_manifest_present': EVAL_MANIFEST_PATH.exists(),
    'working_pool_rows': len(working_pool),
    'working_pool_source_counts': working_pool['dataset_id'].value_counts().to_dict(),
    'final_status_counts': final_status_counts,
    'manifest_summaries': summaries,
    'max_total_tokens': MAX_TOTAL_TOKENS,
    'near_duplicate_threshold': NEAR_DUPLICATE_THRESHOLD,
    'decontamination_threshold': DECONTAMINATION_THRESHOLD,
    'tokenizer_revision': tokenizer_revision,
    'template_hash': TEMPLATE_HASH,
}
with open(REPORT_DIR / f'audit_summary_{RUN_TAG}.json', 'w', encoding='utf-8') as f:
    json.dump(audit_summary, f, indent=2)

manifest_lines = [
    f"- `{name}`: {summary['rows']:,} rows; ordered-ID SHA-256 `{summary['ordered_ids_sha256']}`"
    for name, summary in [('debug_32', debug_summary)] + [(f'train_n{size}', summaries[size]) for size in sorted(summaries)]
]
status_lines = [f'- `{name}`: {count:,}' for name, count in sorted(final_status_counts.items())]
audit_report = [
    f'# Person 1 data audit — {RUN_TAG}', '',
    '## Reproducibility', '',
    f'- Dataset revisions: Orca `{orca_revision}` and Numina `{numina_revision}`',
    f'- Tokenizer: `{TOKENIZER_ID}` at `{tokenizer_revision}`',
    f'- Template SHA-256: `{TEMPLATE_HASH}`',
    f'- Seeds: split `{SPLIT_SEED}`, selection `{SELECTION_SEED}`', '',
    '## Applied controls', '',
    f'- Exact-answer conflicts: quarantined {len(conflicting_row_ids):,} rows.',
    f'- Train near-duplicate threshold: {NEAR_DUPLICATE_THRESHOLD:.2f}.',
    f'- Evaluation decontamination threshold: {DECONTAMINATION_THRESHOLD:.2f}.',
    f'- Maximum rendered sequence length: {MAX_TOTAL_TOKENS:,} tokens.',
    f'- Evaluation manifest supplied: {EVAL_MANIFEST_PATH.exists()}.', '',
    '## Row outcomes', '', *status_lines, '',
    '## Manifests', '', *manifest_lines, '',
    '## Limitations', '',
    f'- Near-duplicate search was limited to {universe_per_source:,} post-deduplication candidates per source.',
    '- Evaluation protection covers only the supplied frozen questions; readiness requires both development families.',
    '- Threshold counts describe conservative lexical variants, not proven identical benchmark instances.',
    '- Quality/review flag counts are nonexclusive. Row outcomes above are mutually exclusive.',
    '- Correctness is unknown; preprocessing token counts do not finalize continuation budget B.',
    '- This process cannot detect exposure through unknown model pretraining data.',
]
(REPORT_DIR / f'data_audit_{RUN_TAG}.md').write_text('\n'.join(audit_report) + '\n', encoding='utf-8')

print('Structural assertions passed. Human review, mathematical correctness and integration approval remain separate.')
print(json.dumps(audit_summary, indent=2))

15 Boundary with Person 2

In [ ]:
PERSON2_TOKEN_FILE = INPUT_DIR / 'person2_token_verification.csv'
if PERSON2_TOKEN_FILE.exists() and PERSON2_TOKEN_METADATA_PATH.exists():
    verification_metadata = json.loads(PERSON2_TOKEN_METADATA_PATH.read_text(encoding='utf-8'))
    if verification_metadata.get('manifest_sha256') != summaries[1000]['sha256']:
        raise ValueError('Person 2 label counts refer to a different R1000 manifest')
    if verification_metadata.get('verification_sha256') != file_sha256(PERSON2_TOKEN_FILE):
        raise ValueError('Person 2 verification file hash mismatch')
    if not verification_metadata.get('confirmed_by') or not verification_metadata.get('confirmed_utc'):
        raise ValueError('Person 2 must confirm actual labels != -100 under the training collator')
    if verification_metadata.get('count_definition') != 'labels != -100 including supervised EOS':
        raise ValueError('Person 2 must use the agreed actual training-label count definition')
    verified = pd.read_csv(PERSON2_TOKEN_FILE, dtype={'row_id': str})
    comparison = reconcile_training_tokens(manifests[1000], verified)
    comparison.to_csv(REPORT_DIR / 'person2_token_reconciliation.csv', index=False)
    TOKEN_LABELS_VERIFIED = True
    VERIFIED_R1000_TOKEN_BUDGET = int(comparison['actual_supervised_tokens'].sum())
    print('R1000 actual training-label counts match; verified budget:', VERIFIED_R1000_TOKEN_BUDGET)
else:
    INTEGRATION_BLOCKERS.append('Person 2 actual R1000 training-label counts/metadata missing')
    print('Waiting for actual label coverage; preprocessing totals remain provisional.')


16 D8 Prepare the fresh continuation candidate pool later

In [ ]:
# Planned templates are run-specific. They are never evidence of what a parent actually saw.
if 1000 in manifests:
    manifests[1000][['problem_group_id']].drop_duplicates().to_csv(
        REPORT_DIR / 'parent_seen_groups_R1000_TEMPLATE.csv', index=False)

if not all(path.exists() for path in [PARENT_SEEN_GROUPS_PATH, PARENT_METADATA_PATH, PARENT_ACTUAL_MANIFEST_PATH]):
    INTEGRATION_BLOCKERS.append('Person 2 actual parent checkpoint, manifest and seen groups missing')
    print('D8 waiting for actual parent confirmation.')
else:
    seen = pd.read_csv(PARENT_SEEN_GROUPS_PATH, dtype=str)
    parent_metadata = json.loads(PARENT_METADATA_PATH.read_text(encoding='utf-8'))
    parent_rows = validate_parent_evidence(seen, PARENT_ACTUAL_MANIFEST_PATH, parent_metadata, summaries[1000]['sha256'])
    # Stable row IDs plus exact problem hashes bridge group IDs across different runs.
    parent_ids = {row['row_id'] for row in parent_rows}
    if any(not row.get('problem') for row in parent_rows):
        raise ValueError('Actual parent manifest needs problem texts for cross-version group protection')
    parent_hashes = {sha256_text(normalize_problem_for_matching(row['problem'])) for row in parent_rows}
    seen_groups = set(seen['problem_group_id'])
    mapped_groups = set(usable_df.loc[
        usable_df['row_id'].isin(parent_ids) | usable_df['problem_hash'].isin(parent_hashes), 'problem_group_id'])
    seen_groups |= mapped_groups
    # Parent data from outside this checked universe require a separate near-overlap pass.
    parent_shingles = [word_shingles(row['problem']) for row in parent_rows]
    parent_lsh = MinHashLSH(threshold=LSH_CANDIDATE_THRESHOLD, num_perm=MINHASH_PERMUTATIONS)
    for index, shingles in enumerate(parent_shingles):
        if shingles: parent_lsh.insert(str(index), make_minhash(shingles))
    parent_near_groups = set()
    for row in tqdm(usable_df.itertuples(index=False), total=len(usable_df), desc='Checking fresh versus actual parent'):
        shingles = word_shingles(row.problem)
        if not shingles: continue
        for hit in parent_lsh.query(make_minhash(shingles)):
            other = parent_shingles[int(hit)]
            union = shingles | other
            if union and len(shingles & other) / len(union) >= NEAR_DUPLICATE_THRESHOLD:
                parent_near_groups.add(row.problem_group_id)
    seen_groups |= parent_near_groups
    fresh_candidates = usable_df[
        ~usable_df['problem_group_id'].isin(seen_groups) & ~usable_df['row_id'].isin(parent_ids)
        & ~usable_df['problem_hash'].isin(parent_hashes)].copy()
    fresh_candidates = balanced_round_robin_order(fresh_candidates, SELECTION_SEED + 1)
    per_source = FRESH_POOL_TARGET // 2
    if FRESH_POOL_TARGET % 2 or any((fresh_candidates['dataset_id'] == source).sum() < per_source for source in ['orca','numina']):
        raise ValueError('Insufficient source-balanced fresh candidates; expand and check the candidate universe')
    fresh_pool = balanced_round_robin_order(pd.concat([
        fresh_candidates[fresh_candidates['dataset_id'] == source].head(per_source) for source in ['orca','numina']
    ], ignore_index=True), SELECTION_SEED + 1)
    assert len(fresh_pool) == FRESH_POOL_TARGET
    assert fresh_pool['problem_group_id'].is_unique
    assert not set(fresh_pool['problem_group_id']) & seen_groups
    assert not set(fresh_pool['row_id']) & parent_ids
    assert not set(fresh_pool['problem_hash']) & (parent_hashes | eval_hashes)
    fresh_out, fresh_summary = save_manifest('fresh_candidate_pool_R1000', fresh_pool)
    fresh_summary.update({
        'parent_checkpoint_id': parent_metadata['checkpoint_id'],
        'parent_checkpoint_sha256': parent_metadata['checkpoint_sha256'],
        'parent_actual_manifest_sha256': file_sha256(PARENT_ACTUAL_MANIFEST_PATH),
        'parent_seen_groups_sha256': file_sha256(PARENT_SEEN_GROUPS_PATH),
        'parent_seen_group_count': len(seen_groups), 'eligible_fresh_candidates': len(fresh_candidates),
        'supervised_length_summary': fresh_pool['supervised_tokens'].describe().to_dict(),
    })
    (MANIFEST_DIR / f'fresh_candidate_pool_R1000_{RUN_TAG}_summary.json').write_text(
        json.dumps(fresh_summary, indent=2), encoding='utf-8')
    PARENT_CONFIRMED = True
    print('Fresh pool exported as DRAFT:', fresh_summary['rows'])


17 Approve the exact corrected handoff using saved review and team evidence


In [ ]:
# Approval is saved evidence bound to this run and exact exported files.
REVIEW_FILES = ['orca_inspection_25.csv', 'numina_inspection_25.csv',
    'orca_answer_extraction_review_25.csv', 'near_duplicate_review_sample.csv',
    'evaluation_near_review_sample.csv', 'missed_match_review.csv', 'formatted_examples_review_20.csv']
review_error = None
review_hashes = None
artifact_hashes = None
try:
    artifact_hashes = verify_export_hashes(
        MANIFEST_DIR, REPORT_DIR, RUN_TAG, summaries, PROMPT_SPEC_HASH,
        fresh_summary=fresh_summary if PARENT_CONFIRMED else None, debug_summary=debug_summary)
    review_hashes = validate_review_approval(REPORT_DIR, REVIEW_FILES, REVIEW_POLICY, RUN_FINGERPRINT, artifact_hashes)
    selected_frames = [working_pool] + ([fresh_pool] if PARENT_CONFIRMED else [])
    validate_review_selection(REPORT_DIR, REVIEW_FILES, REVIEW_POLICY, selected_frames)
except (OSError, ValueError, KeyError) as error:
    review_error = str(error)
    INTEGRATION_BLOCKERS.append('Saved review/export evidence incomplete: ' + review_error)
if not DEVELOPMENT_PROTECTED:
    INTEGRATION_BLOCKERS.append('Frozen development questions have not been protected')
if not TOKEN_LABELS_VERIFIED:
    INTEGRATION_BLOCKERS.append('Actual R1000 labels have not been reconciled')
if not PARENT_CONFIRMED:
    INTEGRATION_BLOCKERS.append('Fresh pool requires confirmed corrected parent evidence')
readiness = {
    'run_fingerprint': RUN_FINGERPRINT, 'research_ready': not INTEGRATION_BLOCKERS and RUN_MODE == 'full',
    'blockers': sorted(set(INTEGRATION_BLOCKERS)), 'development_protected': DEVELOPMENT_PROTECTED,
    'actual_labels_verified': TOKEN_LABELS_VERIFIED, 'parent_confirmed': PARENT_CONFIRMED,
    'verified_R1000_budget': VERIFIED_R1000_TOKEN_BUDGET if TOKEN_LABELS_VERIFIED else None,
    'approved_review_hashes': review_hashes if review_error is None else None,
    'artifact_sha256': artifact_hashes,
    'manifest_sha256': {str(size): summaries[size]['sha256'] for size in summaries},
    'fresh_manifest_sha256': fresh_summary['sha256'] if PARENT_CONFIRMED else None,
    'prompt_specification_sha256': PROMPT_SPEC_HASH,
}
(REPORT_DIR / 'handoff_readiness.json').write_text(json.dumps(readiness, indent=2), encoding='utf-8')
print(json.dumps(readiness, indent=2))
if not readiness['research_ready']:
    raise RuntimeError('DRAFT exports created. Main comparative runs remain gated by handoff_readiness.json.')
print('Saved evidence and integration checks passed for these exact artifact hashes.')
